# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 11/12** (CPU, internet off). **1777 series**, ~10.6 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part11.zip`
- `manifest_rgb25d_320_part11.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 11
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231","series":"1.2.826.0.1.3680043.8.498.12689544351656485342798482846432921455","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542","series":"1.2.826.0.1.3680043.8.498.70243599053060014636616038265618938578","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606","series":"1.2.826.0.1.3680043.8.498.28176392245110944163839399659439535293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559","series":"1.2.826.0.1.3680043.8.498.17927783255029019223687232936481918350","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806","series":"1.2.826.0.1.3680043.8.498.12061584096631077508632129628209972719","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018","series":"1.2.826.0.1.3680043.8.498.33316216096749283413269516635969265980","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401","series":"1.2.826.0.1.3680043.8.498.98396227490752197049907798146084545731","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922","series":"1.2.826.0.1.3680043.8.498.12523349523523873701974348776140572598","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018","series":"1.2.826.0.1.3680043.8.498.40138084701193504692300116021221928305","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365","series":"1.2.826.0.1.3680043.8.498.19687789349338073696932144805224885139","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097","series":"1.2.826.0.1.3680043.8.498.57436347927842361053377226812791686870","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149","series":"1.2.826.0.1.3680043.8.498.10791858932231443712736629517371931282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364","series":"1.2.826.0.1.3680043.8.498.42945613138521535562676526918755318255","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636","series":"1.2.826.0.1.3680043.8.498.68554945431541040670709200116747901472","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882","series":"1.2.826.0.1.3680043.8.498.38478068839340197314332463139684733946","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834","series":"1.2.826.0.1.3680043.8.498.11727353376034732804034873202604308961","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586","series":"1.2.826.0.1.3680043.8.498.18250552347912320116850890825534074677","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471","series":"1.2.826.0.1.3680043.8.498.12349968768708706953429040897225514611","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182","series":"1.2.826.0.1.3680043.8.498.47899976202009359616397788505051078112","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586","series":"1.2.826.0.1.3680043.8.498.69279151696869294053029050274360354721","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221","series":"1.2.826.0.1.3680043.8.498.12465559238103774374376204069568213378","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205","series":"1.2.826.0.1.3680043.8.498.40788347567229295946987885536290671852","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930","series":"1.2.826.0.1.3680043.8.498.52600005062861682762254226830785954060","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161","series":"1.2.826.0.1.3680043.8.498.10726567133672517832047360192933992188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10162720374291180196973998234794414415","series":"1.2.826.0.1.3680043.8.498.13304763566771283298203682646565039471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973","series":"1.2.826.0.1.3680043.8.498.26151363969008758299868119915306542718","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197","series":"1.2.826.0.1.3680043.8.498.12262601433106960284639393383887092405","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10189392162056736163704940747328177364","series":"1.2.826.0.1.3680043.8.498.10857715131994138598011004110668996188","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10197565004992937722841731691775578390","series":"1.2.826.0.1.3680043.8.498.60489697451336587095542334740414876040","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775","series":"1.2.826.0.1.3680043.8.498.10412242176807447471327601873666239732","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10207134048598925821261298801127950516","series":"1.2.826.0.1.3680043.8.498.69357085928739043872700418618329301964","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062","series":"1.2.826.0.1.3680043.8.498.51176597083848383053619356402993534412","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821","series":"1.2.826.0.1.3680043.8.498.13161168622492716861049904953357523966","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576","series":"1.2.826.0.1.3680043.8.498.22903464588641532341502679189979795734","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414","series":"1.2.826.0.1.3680043.8.498.92471874072301720761553891425666714628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531","series":"1.2.826.0.1.3680043.8.498.53853697267386325855050304501825370467","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10254828365033984708960944586564394286","series":"1.2.826.0.1.3680043.8.498.11501661989389720879101040262211766880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10256441455107860474283716992657597873","series":"1.2.826.0.1.3680043.8.498.12499657103871085829545258876350072528","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839","series":"1.2.826.0.1.3680043.8.498.12304370715601989966187711977702706585","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542","series":"1.2.826.0.1.3680043.8.498.24984490234070663176361715753402777480","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639","series":"1.2.826.0.1.3680043.8.498.17867286367390777667851527535179637345","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10284446538566255572004044331831825072","series":"1.2.826.0.1.3680043.8.498.68594777197272736846799007294149623402","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808","series":"1.2.826.0.1.3680043.8.498.92877978003179927499633610841309561294","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10304242666970626179776896768140580106","series":"1.2.826.0.1.3680043.8.498.10716836225584421877601423730579557959","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10310831394568337664174691281092566272","series":"1.2.826.0.1.3680043.8.498.17878732948021367353298797798671059354","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10314240078685804110739316196999571466","series":"1.2.826.0.1.3680043.8.498.63747656240463144163860613478456312396","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548","series":"1.2.826.0.1.3680043.8.498.35789497814262217098604749942496866116","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315","series":"1.2.826.0.1.3680043.8.498.75892971565241529082451849455469914163","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10327608571317884761792196888985963332","series":"1.2.826.0.1.3680043.8.498.85324610001871930187960602576685233561","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726","series":"1.2.826.0.1.3680043.8.498.10355624694647950956997008891539958632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936","series":"1.2.826.0.1.3680043.8.498.29781863298609593653438534606977089434","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10351460911387106879135924409514835385","series":"1.2.826.0.1.3680043.8.498.11088496484440971758808747457138310027","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10352044313309892076985642485759368197","series":"1.2.826.0.1.3680043.8.498.43747241152087917437614395780762274069","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10361030765327974510262894062047216433","series":"1.2.826.0.1.3680043.8.498.93102660447392317964679329254875747621","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218","series":"1.2.826.0.1.3680043.8.498.12629505340899704782067958949831592721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801","series":"1.2.826.0.1.3680043.8.498.12314009451722701931884835374730405401","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341","series":"1.2.826.0.1.3680043.8.498.60137949332330419975614926989658842816","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10391178814963564201551003358372148419","series":"1.2.826.0.1.3680043.8.498.40422448144321049235911137816098490107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007","series":"1.2.826.0.1.3680043.8.498.19956317849367292407114817064618902866","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409","series":"1.2.826.0.1.3680043.8.498.12439486678704691419506069105688882944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667","series":"1.2.826.0.1.3680043.8.498.76086903583936257462304575911841686259","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10419958821528108176389641942168280311","series":"1.2.826.0.1.3680043.8.498.37879332041830925434341172854741834145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689","series":"1.2.826.0.1.3680043.8.498.21694314023612293511309971884062882170","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399","series":"1.2.826.0.1.3680043.8.498.12444202866002271459989864453630808348","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10432568348356281020515782178064869805","series":"1.2.826.0.1.3680043.8.498.12862771526670808608904605078864426170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738","series":"1.2.826.0.1.3680043.8.498.53940935464492939903290813722835027952","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10447920208680921688438738178539475509","series":"1.2.826.0.1.3680043.8.498.91913934045981569591104402380396683755","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674","series":"1.2.826.0.1.3680043.8.498.15390010167165490454975755123634561108","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884","series":"1.2.826.0.1.3680043.8.498.33157186819375788033521204876534341428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10485026284343436524865922231113748546","series":"1.2.826.0.1.3680043.8.498.84805266575991857040521246734101569161","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10492484093480313596330475837639172625","series":"1.2.826.0.1.3680043.8.498.94537004975841218661336487972859324209","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10499265498716562276605625682538326716","series":"1.2.826.0.1.3680043.8.498.86750785011951108457240770736567788192","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10502440129091559437492859441158882078","series":"1.2.826.0.1.3680043.8.498.37113957347329516968707446614943904887","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522","series":"1.2.826.0.1.3680043.8.498.32302325471674978929157248817507623038","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627","series":"1.2.826.0.1.3680043.8.498.87409938310241288529536273683906746126","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10524164196949381411165626176732559196","series":"1.2.826.0.1.3680043.8.498.12284036378535214117815668681189181145","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10532708800468111850607752497432656694","series":"1.2.826.0.1.3680043.8.498.44112310880101671509001224863586298551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10536557774352716497356905292652597953","series":"1.2.826.0.1.3680043.8.498.43839461158863557070322654151423954995","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828","series":"1.2.826.0.1.3680043.8.498.12478996347899137362221957073409956608","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156","series":"1.2.826.0.1.3680043.8.498.11671876921323325996821863019118857275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216","series":"1.2.826.0.1.3680043.8.498.85194099808744912762382183822735027555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10549360192675575195491716398555148512","series":"1.2.826.0.1.3680043.8.498.12465647359487723620905269747682346197","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260","series":"1.2.826.0.1.3680043.8.498.24375078790566504319982468718521198814","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10580260109117947709069163110100632085","series":"1.2.826.0.1.3680043.8.498.72874439418425187246553136429352234899","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10587511485561679190576188107774736852","series":"1.2.826.0.1.3680043.8.498.55328504129708534609935594295450814413","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10590169032156570386146081157598789893","series":"1.2.826.0.1.3680043.8.498.95190800418522950366598080171588707747","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628","series":"1.2.826.0.1.3680043.8.498.26378948108703251678067277220989021967","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146","series":"1.2.826.0.1.3680043.8.498.69550471389646050689415404093373758135","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196","series":"1.2.826.0.1.3680043.8.498.11183129232317754776015535053257951512","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10614508096225944703643482200835890514","series":"1.2.826.0.1.3680043.8.498.33574410857097491507357649739190843401","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430","series":"1.2.826.0.1.3680043.8.498.91918394583103671869473125004804012551","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593","series":"1.2.826.0.1.3680043.8.498.67147134172383321074115855277711753277","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971","series":"1.2.826.0.1.3680043.8.498.13065515215081880112949009003231205554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10632177122947017620482277062226288392","series":"1.2.826.0.1.3680043.8.498.37314878162993638985236359012387104817","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10648346793256473462304051626410900340","series":"1.2.826.0.1.3680043.8.498.38445362843202686905676355260114453820","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658","series":"1.2.826.0.1.3680043.8.498.92301965040673847368938292031809966558","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996","series":"1.2.826.0.1.3680043.8.498.82529458988694896780725754232146300968","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10657584461509809047185519742966035356","series":"1.2.826.0.1.3680043.8.498.12011749173321638055469427977112035959","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10661650555947788912511478165000737040","series":"1.2.826.0.1.3680043.8.498.51224367562544770550607778759918101421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331","series":"1.2.826.0.1.3680043.8.498.13707218513279974084514389054250971924","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071","series":"1.2.826.0.1.3680043.8.498.29724646835497091385876327908204826215","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10688670034893539180306173203903350286","series":"1.2.826.0.1.3680043.8.498.11087830279382205116812159981966865212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080","series":"1.2.826.0.1.3680043.8.498.10247344719939815370295937915057942701","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10715956446507906169485719013515169244","series":"1.2.826.0.1.3680043.8.498.24798410035008900772717543940339253672","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10728448245470003618625208276233778895","series":"1.2.826.0.1.3680043.8.498.79543656724516182383626696358163554865","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171","series":"1.2.826.0.1.3680043.8.498.84047126631088959343658030351228929200","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802","series":"1.2.826.0.1.3680043.8.498.32663356838530160581130731220309075693","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375","series":"1.2.826.0.1.3680043.8.498.72798549999446834141956300840701994399","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10751624265599812767108610284996090628","series":"1.2.826.0.1.3680043.8.498.28377263772047792004747456662257918248","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10759288702627176691392048851622294054","series":"1.2.826.0.1.3680043.8.498.10490278258578942118557141257163634182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762","series":"1.2.826.0.1.3680043.8.498.13276079887735094149760947592710931148","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428","series":"1.2.826.0.1.3680043.8.498.71012884455584456285731916536466396994","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10774147135673365265606645781662088745","series":"1.2.826.0.1.3680043.8.498.93820798503521351221177019227576200469","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334","series":"1.2.826.0.1.3680043.8.498.64967567266633282681675652949828527982","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10802653832050736637820225444196833450","series":"1.2.826.0.1.3680043.8.498.12050968613942881103724996864857518396","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10808663384878038371710260386807468588","series":"1.2.826.0.1.3680043.8.498.90624228984443699667343944635147592006","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10824420630965162625242888916805538597","series":"1.2.826.0.1.3680043.8.498.13327591817198000962574347264212834092","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10830288389793651695251406408666084306","series":"1.2.826.0.1.3680043.8.498.11105976745651525422376314029869294458","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10839126901366636822676830560774505865","series":"1.2.826.0.1.3680043.8.498.10634009940138707006767243395628345306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972","series":"1.2.826.0.1.3680043.8.498.37644664991855014158089945476229092000","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10851562687641986545442408470139368318","series":"1.2.826.0.1.3680043.8.498.10937547719069221267007441455052995673","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10854012268828835845581966634225522931","series":"1.2.826.0.1.3680043.8.498.10098678002281526260238470652261477534","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655","series":"1.2.826.0.1.3680043.8.498.60821300866226604938624568275240480914","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10862481204237376580557516731952961213","series":"1.2.826.0.1.3680043.8.498.63364197589900765731023792551632908470","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10880553082640412137979450100928382021","series":"1.2.826.0.1.3680043.8.498.55372591018362894843781034088255566480","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10886583574130697346788286647281863932","series":"1.2.826.0.1.3680043.8.498.87654258751019577525602621436253132768","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673","series":"1.2.826.0.1.3680043.8.498.10736132749531087097079811480530448777","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10917389623891555306584069980815338466","series":"1.2.826.0.1.3680043.8.498.11766429380821047008279159287631909185","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10918580785296934228715514366408269433","series":"1.2.826.0.1.3680043.8.498.12608533145340432493646814911000771508","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10924685588962763768106617368647678902","series":"1.2.826.0.1.3680043.8.498.95504058673907880702281526921325106885","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359","series":"1.2.826.0.1.3680043.8.498.95240512779611655208131014133044460635","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737","series":"1.2.826.0.1.3680043.8.498.21611554313273187540092070881454423375","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768","series":"1.2.826.0.1.3680043.8.498.87876383576270611983952602016146459024","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626","series":"1.2.826.0.1.3680043.8.498.10644445117306494205287833202529606508","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670","series":"1.2.826.0.1.3680043.8.498.79543990040716037715501701971047575510","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575","series":"1.2.826.0.1.3680043.8.498.10263657348351241577341013406449965514","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10975347305662629858093628672419901528","series":"1.2.826.0.1.3680043.8.498.13304791872982909151644015575017945475","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10981062990502278795793220223076241424","series":"1.2.826.0.1.3680043.8.498.77936382170028111223556489433975145227","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699","series":"1.2.826.0.1.3680043.8.498.11193344262215846482320197273402127392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174","series":"1.2.826.0.1.3680043.8.498.63009377438116990863303189359667886290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480","series":"1.2.826.0.1.3680043.8.498.70610524876150888552633131657190533763","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11006241238894862275805535376070111510","series":"1.2.826.0.1.3680043.8.498.11561243623444275322494059986132344511","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11010004366885474301784150582039137111","series":"1.2.826.0.1.3680043.8.498.53586115095368197321625265173708372517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310","series":"1.2.826.0.1.3680043.8.498.71585447033365787912519640179748543872","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043081829401721095803441219553837608","series":"1.2.826.0.1.3680043.8.498.61773350923869047257309487645987988538","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11045459770905584149950259201414074545","series":"1.2.826.0.1.3680043.8.498.85075200115945471144727546716581899447","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11052656375391331522834364601050945860","series":"1.2.826.0.1.3680043.8.498.10391936951859709862624758388224646150","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11060834081463491262807709963877480026","series":"1.2.826.0.1.3680043.8.498.38047728237172508061947656258055375584","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11067200656721609390997819562401839124","series":"1.2.826.0.1.3680043.8.498.77174146563970283436603471770263283736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11079556243110299049578981210837702995","series":"1.2.826.0.1.3680043.8.498.12527125077522644237077328932898958400","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11084010833221855396433326769153950681","series":"1.2.826.0.1.3680043.8.498.18489043351575480905003206899953535754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11100224438877567316062152508871079233","series":"1.2.826.0.1.3680043.8.498.71650390879904615791922843410911162965","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11110575656494405149601490027444484170","series":"1.2.826.0.1.3680043.8.498.10951965265526730765527211950753407636","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11119324966854127911591363740049886324","series":"1.2.826.0.1.3680043.8.498.12724132064614845209694019224901675557","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11123446342116837882811834332727990911","series":"1.2.826.0.1.3680043.8.498.78428842255726508682348840339220652883","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11126383719852491587081347886000714259","series":"1.2.826.0.1.3680043.8.498.20005803380697261754983677821984833473","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11131210140372132238166068091728877219","series":"1.2.826.0.1.3680043.8.498.30509135361619436572012385033676393841","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286","series":"1.2.826.0.1.3680043.8.498.69918162033971502597110456336170542407","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11148939654362681630811787929668902046","series":"1.2.826.0.1.3680043.8.498.44524319115442236769067331476859301871","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11150193626240150189232364321568347937","series":"1.2.826.0.1.3680043.8.498.11167260759049759079709904667320379831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11152585968424897353533125085307104033","series":"1.2.826.0.1.3680043.8.498.14592543121470926536424003831079712400","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11156884483511690263238401598901614096","series":"1.2.826.0.1.3680043.8.498.46208887186138795149206878273927492535","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11180457573346204459956051956213528997","series":"1.2.826.0.1.3680043.8.498.10819869988943545428834611126030555920","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11184968672176462852561645279750399747","series":"1.2.826.0.1.3680043.8.498.60051488317192457439759427477111276587","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11192287547210248515805146368662883131","series":"1.2.826.0.1.3680043.8.498.12410998422105889041230225671656091598","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878","series":"1.2.826.0.1.3680043.8.498.16535880882710155944588846038842258521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","series":"1.2.826.0.1.3680043.8.498.55932695006796933644996573437261464366","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11211598263863419448236436640277320409","series":"1.2.826.0.1.3680043.8.498.13194750573601603607536192344416623678","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11218351123386367961865331646414911627","series":"1.2.826.0.1.3680043.8.498.57848819544572419034653596365327007193","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11219813299300915347039206446596901205","series":"1.2.826.0.1.3680043.8.498.13260956925682693091443661638038367291","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11226582925516370364253861721606580867","series":"1.2.826.0.1.3680043.8.498.52791145452482584046666716253339080521","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11231822445970527276203191047278744341","series":"1.2.826.0.1.3680043.8.498.21357004784570267064738433710120268191","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11244115315611630362929576340631670750","series":"1.2.826.0.1.3680043.8.498.13839578070366157889303677859993927876","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11257168136762043265165109052826442974","series":"1.2.826.0.1.3680043.8.498.65106503824052378939036647989838489667","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683","series":"1.2.826.0.1.3680043.8.498.65553596642852645477065357156416656979","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11268635423021790546063903457498005204","series":"1.2.826.0.1.3680043.8.498.14484402814440464137551342394390166836","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11275711698468745807758222802302685022","series":"1.2.826.0.1.3680043.8.498.11113709449952490169005248485985230710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11277078989123345865112116288626381353","series":"1.2.826.0.1.3680043.8.498.51538385517383307709896505660900968131","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11282706861378379965012380644125662529","series":"1.2.826.0.1.3680043.8.498.79445614773782091506926205958153058383","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580","series":"1.2.826.0.1.3680043.8.498.11454182648928474635381502957910523653","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11294885816514924203337081832122619605","series":"1.2.826.0.1.3680043.8.498.45977835813484597706832123660553037583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11298099571581989006135838942599921393","series":"1.2.826.0.1.3680043.8.498.13254265968449604690981924266256171777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11307870660160563329996058267938398411","series":"1.2.826.0.1.3680043.8.498.11153486445207634741842745756157292036","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616","series":"1.2.826.0.1.3680043.8.498.67765663587327532607624409861928609133","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11331824020096159502562313322049966712","series":"1.2.826.0.1.3680043.8.498.11831466582072149886255868654603215611","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11334776960319816844935217444817103072","series":"1.2.826.0.1.3680043.8.498.21538722495588902836315143471400375187","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11340429352365972769264064439788186260","series":"1.2.826.0.1.3680043.8.498.44789160653676253469273608243955976358","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11352722814008170661344291337061231555","series":"1.2.826.0.1.3680043.8.498.12668585944578825530108813337675332159","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11362547519125634779422991321337283241","series":"1.2.826.0.1.3680043.8.498.10147107332860035200909994376691719772","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238","series":"1.2.826.0.1.3680043.8.498.14985374264812063212398702944383827602","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994","series":"1.2.826.0.1.3680043.8.498.24725345037283573415439855171951180329","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11394861774586578112386386342670500749","series":"1.2.826.0.1.3680043.8.498.20952275945492602303541749665244110869","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11396451979734570993257791565721573849","series":"1.2.826.0.1.3680043.8.498.11956279797109758813029179104108642764","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11406381777983128446583783158882827226","series":"1.2.826.0.1.3680043.8.498.41422746277627294116497200998469112883","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11420670844338686032030860787688858057","series":"1.2.826.0.1.3680043.8.498.98739815249429539011863338701701369729","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11425683153901568541658945026861110110","series":"1.2.826.0.1.3680043.8.498.11498760768351837751129231916097023006","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11432815311029789072253610823554308183","series":"1.2.826.0.1.3680043.8.498.73622998013540223372038585897105183121","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11441489276425334980205891153179893797","series":"1.2.826.0.1.3680043.8.498.49340834068230197734784787185409549788","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579","series":"1.2.826.0.1.3680043.8.498.83611671629896300140933311161835210413","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11451981256762378203455481812714370193","series":"1.2.826.0.1.3680043.8.498.40339326189295583375029477908865711423","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11458488293461896930750018852127694875","series":"1.2.826.0.1.3680043.8.498.64402169798087333509554406556421463285","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11465586454035959390661499120751232686","series":"1.2.826.0.1.3680043.8.498.35275627587044578602950791719732295150","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11471148332557464549579838595592736047","series":"1.2.826.0.1.3680043.8.498.11842001789948198975786614425334713375","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11477000011453869751231224714283666904","series":"1.2.826.0.1.3680043.8.498.79080307107333842850552427612247623274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719","series":"1.2.826.0.1.3680043.8.498.98923465417715022887084745967335046843","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434","series":"1.2.826.0.1.3680043.8.498.86985551635887653095455398165431767796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498505068787084440256890724559761815","series":"1.2.826.0.1.3680043.8.498.22288958242328317250995765702268488174","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11501070108039994411346060488666341726","series":"1.2.826.0.1.3680043.8.498.52942263321999045606782542499041429107","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11506856887791286395737465774061395929","series":"1.2.826.0.1.3680043.8.498.12437229684036318377424276688009212263","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664","series":"1.2.826.0.1.3680043.8.498.31584070441764492778800366759705001471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11516842786271998149681118182571061813","series":"1.2.826.0.1.3680043.8.498.57634447376915220162935675209580883175","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11529405479134457001825038259936796388","series":"1.2.826.0.1.3680043.8.498.10663522714217703523859594064553077228","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11533971444023236216745801026152089075","series":"1.2.826.0.1.3680043.8.498.63582955966333706107742845130652643286","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11543622448213658856759993374276266341","series":"1.2.826.0.1.3680043.8.498.22100004123650363936549328302260287698","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11547192037231064935759879673500408149","series":"1.2.826.0.1.3680043.8.498.66609753611291405796993584727683317421","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11548839502674800051973767797346682106","series":"1.2.826.0.1.3680043.8.498.25907249512833959586971692392537049502","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840","series":"1.2.826.0.1.3680043.8.498.46111989241956497589129886237673737566","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11569355546217846610131708883985793793","series":"1.2.826.0.1.3680043.8.498.79240019274187933792578935356797109245","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11574480940878399452466699512074769686","series":"1.2.826.0.1.3680043.8.498.75269279898263232161761433605018360629","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11577630811395858629711985689258647804","series":"1.2.826.0.1.3680043.8.498.57557016066015267782394467181591651529","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552","series":"1.2.826.0.1.3680043.8.498.12599934826555786383316422707329168897","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11590096889491409910425456896187293515","series":"1.2.826.0.1.3680043.8.498.61759688462134644234242615752482396743","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11592838054859712596515534341207971997","series":"1.2.826.0.1.3680043.8.498.44190380981011916830058629570153772703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11596665780617412984766822528862741584","series":"1.2.826.0.1.3680043.8.498.11169158928779880798108509329821926091","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11599387556469808488160280438305325860","series":"1.2.826.0.1.3680043.8.498.10739992981632503167285492161563723627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11605266731872772432220415835172557961","series":"1.2.826.0.1.3680043.8.498.16884644212138500218596055738487531441","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11610585947047502123824835405421129552","series":"1.2.826.0.1.3680043.8.498.11159120959471023026993735239702011942","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378","series":"1.2.826.0.1.3680043.8.498.75139601678338667489704572802585495595","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11641274367861998748475337224297076397","series":"1.2.826.0.1.3680043.8.498.10944865160560979081957974719354643639","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11642228346306175407288158335830376197","series":"1.2.826.0.1.3680043.8.498.74509109297784341428294589321161791869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11647595587705638372290862475518195181","series":"1.2.826.0.1.3680043.8.498.12855989985655999505495145272107824937","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11652570369213315569275956969935617843","series":"1.2.826.0.1.3680043.8.498.96832375822372017531066290669279596022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093","series":"1.2.826.0.1.3680043.8.498.13012952090586376177238554110051488611","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11669120316759278775792190599036288128","series":"1.2.826.0.1.3680043.8.498.26488967990512921120133618718595473915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11679981444785137570894759919380932611","series":"1.2.826.0.1.3680043.8.498.75595084019894820076187901275594764993","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11685904538570360040472064238371379681","series":"1.2.826.0.1.3680043.8.498.54519972773665937111894855843418180872","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11691245281415120685024733512803575288","series":"1.2.826.0.1.3680043.8.498.42601900544539152801710048303482572988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11706378290185110677863653788588307917","series":"1.2.826.0.1.3680043.8.498.78586897505768927631514873239488959927","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11716730437469973434747875401501334429","series":"1.2.826.0.1.3680043.8.498.39095862822423225171502337066434731574","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11720535027022778971695130536590302635","series":"1.2.826.0.1.3680043.8.498.57106282374263218891723636432873656156","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612","series":"1.2.826.0.1.3680043.8.498.40436189400403161513802840351544666595","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11725843001854130477287484865097535511","series":"1.2.826.0.1.3680043.8.498.10761939823872938471871706427455239702","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11751697254880722008245447197446027607","series":"1.2.826.0.1.3680043.8.498.33048800036292905749401156507392284296","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11762352496600661264881556836248251400","series":"1.2.826.0.1.3680043.8.498.64553649286034703233270750935539955370","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11770690655120964721059321709525979950","series":"1.2.826.0.1.3680043.8.498.61931553595536312431382776495464220628","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11776647434143676427722616249587614803","series":"1.2.826.0.1.3680043.8.498.14538770595197089133238939177874256551","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11785138333495400389510795613252286069","series":"1.2.826.0.1.3680043.8.498.32492617860557613632099634248982925004","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11786582806577871618290342375465595269","series":"1.2.826.0.1.3680043.8.498.10484704639753735091712441075498733847","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11794753699425693678056056340551238033","series":"1.2.826.0.1.3680043.8.498.22280947285766785120771364123892081187","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11798183157109435753679723596499976762","series":"1.2.826.0.1.3680043.8.498.10987377836642751711530384619196394627","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11803762390999881908101130422708589537","series":"1.2.826.0.1.3680043.8.498.11473723819177118282426741916734147534","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991","series":"1.2.826.0.1.3680043.8.498.12944903542719919260187384437623728064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11814923363609140180293604689814785969","series":"1.2.826.0.1.3680043.8.498.56043416714004501901306063583526426548","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11830915612184936576804778995164042200","series":"1.2.826.0.1.3680043.8.498.83930584189633695372764182855574596447","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11851285389689341495961233129719325417","series":"1.2.826.0.1.3680043.8.498.10549882504934137239946205948987958619","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11863890747360341056211511874563792862","series":"1.2.826.0.1.3680043.8.498.84400464023232598434227952000666402254","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697","series":"1.2.826.0.1.3680043.8.498.58521247172333872164251603756153410248","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","series":"1.2.826.0.1.3680043.8.498.35637777476707659871659169588184134922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11881735311125899170335963782017197184","series":"1.2.826.0.1.3680043.8.498.53976076374258651270497886928398924380","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11892908237449514945544380023820626319","series":"1.2.826.0.1.3680043.8.498.11931728732211899159980707076717933544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11901474252960403779405194385221103673","series":"1.2.826.0.1.3680043.8.498.12295929763226378522182911174496860158","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369","series":"1.2.826.0.1.3680043.8.498.33231897399023621668807688285779070948","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11909779134152823550896593167929611759","series":"1.2.826.0.1.3680043.8.498.88717457117883928145971440082662923568","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11912618147674655465071922267060032851","series":"1.2.826.0.1.3680043.8.498.11914354743578042990183571808547572729","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11919420571588879483419578078755803351","series":"1.2.826.0.1.3680043.8.498.10792399883012761025270748584263180631","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11923218013055841952315996726408812182","series":"1.2.826.0.1.3680043.8.498.75845080461638849813657848365297011106","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028","series":"1.2.826.0.1.3680043.8.498.13019765955406124790508104724063213433","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890","series":"1.2.826.0.1.3680043.8.498.11217055425962450114317672996174603331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11937911933281010146897685679862930140","series":"1.2.826.0.1.3680043.8.498.91351276369320632625074185755681234272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769","series":"1.2.826.0.1.3680043.8.498.57713304595268187801277158386014982545","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11957476321850340968244508976719889277","series":"1.2.826.0.1.3680043.8.498.59448122125493399897964180771803432364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317","series":"1.2.826.0.1.3680043.8.498.73295857871152999594050476721545261592","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11979906027101464432999416052202616662","series":"1.2.826.0.1.3680043.8.498.10316447045175393554621934984365782335","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11980805687949950504930975019161587970","series":"1.2.826.0.1.3680043.8.498.88508209423883286521220167855731363979","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11991512352474459753052051071717622771","series":"1.2.826.0.1.3680043.8.498.35874434111160382718705038648317081598","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11999008448480965933785988718780864948","series":"1.2.826.0.1.3680043.8.498.85820928530872230625756769173447722815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12007335335641704192494936946829650792","series":"1.2.826.0.1.3680043.8.498.29371746826583094116091495848734784482","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084","series":"1.2.826.0.1.3680043.8.498.14271391565259762819206539874788476502","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12026457191304650188547548517205757122","series":"1.2.826.0.1.3680043.8.498.11429093389811861944713411718253948585","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12034579945815402722591687609951730205","series":"1.2.826.0.1.3680043.8.498.13259953553004679221080997083143158511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12041392088066705782960819598258792338","series":"1.2.826.0.1.3680043.8.498.34342562388357782561214029231076586481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179","series":"1.2.826.0.1.3680043.8.498.50699047078402990941500110970555763384","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12054099347723802635102026371218576198","series":"1.2.826.0.1.3680043.8.498.10560148514295836473281434254568913454","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12067625141402105065337444349201280606","series":"1.2.826.0.1.3680043.8.498.72422113105235936909533893307228413516","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12071998470084955703027522152040042364","series":"1.2.826.0.1.3680043.8.498.11626521365788057179578334210164694944","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12095779329738382018467257299597656491","series":"1.2.826.0.1.3680043.8.498.14835724122036191036666535627681559354","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515","series":"1.2.826.0.1.3680043.8.498.84839789536079013731735167942481940814","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12107680313133383587522762626929218051","series":"1.2.826.0.1.3680043.8.498.82551368487071003986034490496373469852","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12130605331118953242364684736005454581","series":"1.2.826.0.1.3680043.8.498.73756507469241365440224847668747340880","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12132814192991105841939278718608095213","series":"1.2.826.0.1.3680043.8.498.12552431439683834741331559006974197171","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12137739162059836759686062882514824614","series":"1.2.826.0.1.3680043.8.498.49682950151330220157828024881635298033","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267","series":"1.2.826.0.1.3680043.8.498.58612354331585082785673256031709382393","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12154188739734505793699974626991872621","series":"1.2.826.0.1.3680043.8.498.19348630980811267327210117842073598018","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12165893434332227221101266719291133169","series":"1.2.826.0.1.3680043.8.498.32348487506180031863460637036442120438","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12166235678892893545054546864889743251","series":"1.2.826.0.1.3680043.8.498.10640566728857990597420814346747378035","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12171469604213841534511019872648327851","series":"1.2.826.0.1.3680043.8.498.91733952044851572467645605647330930755","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12176466551281285838907773521868954581","series":"1.2.826.0.1.3680043.8.498.12731338636677261325511791627759359543","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211","series":"1.2.826.0.1.3680043.8.498.11095676664461968627994537962137028040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12185615973620890770158935333033805789","series":"1.2.826.0.1.3680043.8.498.28758729699326529919928575126280761202","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408","series":"1.2.826.0.1.3680043.8.498.27630079257751312816068733933059642988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12195090472992304712555822541659656360","series":"1.2.826.0.1.3680043.8.498.20624400107035562940776897688360333749","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12202865491511254130359509254373393681","series":"1.2.826.0.1.3680043.8.498.10638001299584095987610489929510929782","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968","series":"1.2.826.0.1.3680043.8.498.11379254154882764513568812284249542939","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12221892384948220776502422531414659289","series":"1.2.826.0.1.3680043.8.498.50944244605347020475788763096816895862","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465","series":"1.2.826.0.1.3680043.8.498.21340183407243581025971975668173387056","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615","series":"1.2.826.0.1.3680043.8.498.19187409021176611810456388383623798324","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494","series":"1.2.826.0.1.3680043.8.498.12479207878945563513431080447285049395","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12245760386130805526409111049874664817","series":"1.2.826.0.1.3680043.8.498.40908069384447668071042220358146174136","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392","series":"1.2.826.0.1.3680043.8.498.65096806654023229078502591993741996805","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12257119852150088212071954292747252243","series":"1.2.826.0.1.3680043.8.498.10808937242141629594849465610716680679","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12260661730837055872432103268905182876","series":"1.2.826.0.1.3680043.8.498.57075596180887157678820299381236398521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12264607993101717201407955391437731134","series":"1.2.826.0.1.3680043.8.498.65287066394973657808617314266381023251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266644932420958618394322959047940542","series":"1.2.826.0.1.3680043.8.498.58365719883492083392321983204868478489","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12271616365050313084031931009756086606","series":"1.2.826.0.1.3680043.8.498.79071153667766966109864825287944670631","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12274801503555668613400469130773239683","series":"1.2.826.0.1.3680043.8.498.10949827780812185912376852439459384406","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12280407492991430179266369843997653205","series":"1.2.826.0.1.3680043.8.498.10850700521627659349008319086401583147","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12285177271228289867211553575739205132","series":"1.2.826.0.1.3680043.8.498.11496985612579910123565911554608752040","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12294073960417956629256115458135748872","series":"1.2.826.0.1.3680043.8.498.64574556400469413191363382923902313966","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12295443776523723913480216844110882937","series":"1.2.826.0.1.3680043.8.498.62589164593010122029163317745036742436","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12300749089664733313051543117302244508","series":"1.2.826.0.1.3680043.8.498.48337211187316510957351471327975827048","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12305281328019552390691110900015627668","series":"1.2.826.0.1.3680043.8.498.67069605533786055425218038109162076656","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12307356432736970350794033171174403632","series":"1.2.826.0.1.3680043.8.498.11621939040639910179117520978307642924","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097","series":"1.2.826.0.1.3680043.8.498.48211487117914810025871703602150968008","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12331841096593105748526634055968654914","series":"1.2.826.0.1.3680043.8.498.53715200184574207313725155678653739696","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081","series":"1.2.826.0.1.3680043.8.498.64397550837394550973627521940538609589","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12347007036329783363431172840583325937","series":"1.2.826.0.1.3680043.8.498.61195853512401951698816873691506701023","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12348875283993064999589090039392140219","series":"1.2.826.0.1.3680043.8.498.81075036384306462837841851712775285671","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12352132666379687393012255589355771387","series":"1.2.826.0.1.3680043.8.498.50892369808073656029842526091253051529","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12355011301785011370279005103251815187","series":"1.2.826.0.1.3680043.8.498.12087797894403626673894728406073856001","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12365033663152692616611548717152384269","series":"1.2.826.0.1.3680043.8.498.11398515408604651205013152395358112235","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12374183329974116022606010915169117790","series":"1.2.826.0.1.3680043.8.498.83990962271686553923285172109360127163","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12379570725733005682132316461218878636","series":"1.2.826.0.1.3680043.8.498.10507365085603578123729615425987725247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12381909868368799280462187344958674397","series":"1.2.826.0.1.3680043.8.498.53927282299890996683318603925211102998","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387075993162655781372091860026675548","series":"1.2.826.0.1.3680043.8.498.70394379913022393758156294597638728157","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387667155732116394489547491673280062","series":"1.2.826.0.1.3680043.8.498.66551848658399543045279734288246623183","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12406043058701411628555010402350032739","series":"1.2.826.0.1.3680043.8.498.12368183489328271709479766943106664216","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408314059030096510457440879771212020","series":"1.2.826.0.1.3680043.8.498.53355126183555571144369590273360958624","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408945963789246646229147558387793270","series":"1.2.826.0.1.3680043.8.498.13398946876177883592322906970570308424","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12414869188691502885668778427120766321","series":"1.2.826.0.1.3680043.8.498.12684000687650317224674751095715662651","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12432974885645948103184031984290883090","series":"1.2.826.0.1.3680043.8.498.13173272080297726117037622047413321837","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12438369033802887819851319347622787876","series":"1.2.826.0.1.3680043.8.498.11598687125748625892340310140434854086","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945","series":"1.2.826.0.1.3680043.8.498.12643342492475545314010779881304326087","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12462498156569316578044317782899399656","series":"1.2.826.0.1.3680043.8.498.11116380736584609872366629559581714670","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12473533314363654795650919604551685896","series":"1.2.826.0.1.3680043.8.498.87881650287385497204741353923109414830","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12481913908649439189738319460342358360","series":"1.2.826.0.1.3680043.8.498.99689985593412192688376667949616844599","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12487362549907644519947672428024977201","series":"1.2.826.0.1.3680043.8.498.63953186418078473846867295168036979519","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12490034912643658601552092574630438642","series":"1.2.826.0.1.3680043.8.498.10426400677829043162223214318928055052","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491747080684524905643593142141672951","series":"1.2.826.0.1.3680043.8.498.47070587378183341234389378788109076361","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12492431885298164225672926785397151525","series":"1.2.826.0.1.3680043.8.498.11675738323217484850396187769017637957","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12501925425699863373513819814520995919","series":"1.2.826.0.1.3680043.8.498.72932823006360301886436053718561384526","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12507957850103916926719601413279798696","series":"1.2.826.0.1.3680043.8.498.26436136520404763932959480889339873131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12510856755988712246013743789183840748","series":"1.2.826.0.1.3680043.8.498.27937524668846222942750847813911487330","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12518297594498173981937681848554641851","series":"1.2.826.0.1.3680043.8.498.64937125249666305919316238178763568148","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12522620957277632939731546099134770342","series":"1.2.826.0.1.3680043.8.498.34891777343395877658103348757602467190","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12526590714362529606046044104426807158","series":"1.2.826.0.1.3680043.8.498.59651359429280484767980331482098775148","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12531442175319757740745664049001806494","series":"1.2.826.0.1.3680043.8.498.88324549979475788416637844089242325493","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12534628800787651008876070514080610643","series":"1.2.826.0.1.3680043.8.498.10033823632938838497542646813749488629","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12552701911592685137269544109687472988","series":"1.2.826.0.1.3680043.8.498.35969117659242762355620069689845725121","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12553455350653672209409408595948190982","series":"1.2.826.0.1.3680043.8.498.43047475598279996138731702872576045365","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12563940497360871791097093821926086616","series":"1.2.826.0.1.3680043.8.498.95879758559786616917463196093174626500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12569762036474456642024618586300354737","series":"1.2.826.0.1.3680043.8.498.11564530057987457184395918847941248621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12573518342767030025851169814440863785","series":"1.2.826.0.1.3680043.8.498.60804107113192247457385607302909820498","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12577828789206467631166875572423539298","series":"1.2.826.0.1.3680043.8.498.40428847949037170280316872366979925878","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12580232188757225504172709417466515184","series":"1.2.826.0.1.3680043.8.498.22134575882004981137927401588931449167","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12606136253498337506682547306933254691","series":"1.2.826.0.1.3680043.8.498.29420598173399898471807075158001264762","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607203615180268117341271176820357242","series":"1.2.826.0.1.3680043.8.498.28523586288915059762662248308970052791","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12614097867016983076979065551985251584","series":"1.2.826.0.1.3680043.8.498.57507782175956024555337864570372416737","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12619895582462244920124420363095282842","series":"1.2.826.0.1.3680043.8.498.86084796069969034368721293669436299756","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12626383124142146001532197239671686442","series":"1.2.826.0.1.3680043.8.498.13607054477291319452900153983438061089","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","series":"1.2.826.0.1.3680043.8.498.56033765375471384389795406877209247994","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12637591861595747322913829625646935491","series":"1.2.826.0.1.3680043.8.498.89241652174698566680161357365212838775","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12640439197386154384526090653248054582","series":"1.2.826.0.1.3680043.8.498.10385351092943012608128418156627192950","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12647791319443598318497312837147189300","series":"1.2.826.0.1.3680043.8.498.40115377772925115967439070844490450394","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12655324218817524683138594937984186560","series":"1.2.826.0.1.3680043.8.498.61145365295126053745384509063359701632","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12668708646283106269005674386248135220","series":"1.2.826.0.1.3680043.8.498.87854405067018421472666923952005822399","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12674069717230126880288353271062347095","series":"1.2.826.0.1.3680043.8.498.89736994507040868045161367891105238587","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12688954493547892047021455671738310186","series":"1.2.826.0.1.3680043.8.498.46950989271410720195859196430090686583","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12690080124553846006638565600585469969","series":"1.2.826.0.1.3680043.8.498.88762965092087700134563504422734589056","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12697859103448296208701081683487338111","series":"1.2.826.0.1.3680043.8.498.51082966618953076938149780099536056685","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12700591955254168977239658917260628326","series":"1.2.826.0.1.3680043.8.498.12259637805167211593153902175326299229","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12704063145595847725421427571860589471","series":"1.2.826.0.1.3680043.8.498.52192545566017964663356374523790782027","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715130507546263968763238702574191950","series":"1.2.826.0.1.3680043.8.498.21979391699792928010016056103172611725","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12717728703860353055063551162640833849","series":"1.2.826.0.1.3680043.8.498.17080917528521585457629025383748081251","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405","series":"1.2.826.0.1.3680043.8.498.13181087873656237107501323209299849015","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12725832336445568031849006413563892122","series":"1.2.826.0.1.3680043.8.498.99833755991364228853682788719832915355","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12728397615513347615111391662798012012","series":"1.2.826.0.1.3680043.8.498.12547673966754789121163080333171237489","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12754381146006839341912662615844567067","series":"1.2.826.0.1.3680043.8.498.12977209316786372591683222443101510749","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12762076104929080571464647825161185643","series":"1.2.826.0.1.3680043.8.498.29583669727727132699926000549694592708","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12765434812409174436720333864278447770","series":"1.2.826.0.1.3680043.8.498.44448275905632095457013605600817493866","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535","series":"1.2.826.0.1.3680043.8.498.51020481064310704933530210972010327785","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775105605170844918620305197829216665","series":"1.2.826.0.1.3680043.8.498.88111636932053938014202881225522068446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180","series":"1.2.826.0.1.3680043.8.498.98853376054514912692254778656988398961","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12790542912560396272981117250277568532","series":"1.2.826.0.1.3680043.8.498.41132923822423111919674540447522140125","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12798419553759200632295966180228523212","series":"1.2.826.0.1.3680043.8.498.11547561702788322971316624452361789020","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12800130397185143653180509040746675358","series":"1.2.826.0.1.3680043.8.498.11805779128990728018332761183126635613","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12806792356922865929896103306611295151","series":"1.2.826.0.1.3680043.8.498.90753002303842822534224496307653115206","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12814598731776085957663058326979908652","series":"1.2.826.0.1.3680043.8.498.13254229739287260368977144045896256055","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12818010040786942613541254450332754046","series":"1.2.826.0.1.3680043.8.498.10285340908954640777948702979779134311","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12820051821104182459960036809430047569","series":"1.2.826.0.1.3680043.8.498.12880014772147988389480668679088372441","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12827120140904847104302376671076396695","series":"1.2.826.0.1.3680043.8.498.10456535892185459781634996988207730458","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12837030643153472711780894512565629507","series":"1.2.826.0.1.3680043.8.498.12462547529658360636926976194428342584","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12862782592303338109631225504988756108","series":"1.2.826.0.1.3680043.8.498.76078071032350690325135559487062887908","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12887544771002132349592859410979448655","series":"1.2.826.0.1.3680043.8.498.10104539774726293870844986361799697620","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12899995703999073968090810947564075608","series":"1.2.826.0.1.3680043.8.498.13072921948442217174166627157789359690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12912436233176216254115594260643600155","series":"1.2.826.0.1.3680043.8.498.38688701975173494850421439714152699275","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664","series":"1.2.826.0.1.3680043.8.498.72615453034079438181782477827356188571","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265","series":"1.2.826.0.1.3680043.8.498.12425365367662767797065142147538604226","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12938500386086171684983091332136233036","series":"1.2.826.0.1.3680043.8.498.64378235991462857289117855137078342893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12946789387115203718210904958275579544","series":"1.2.826.0.1.3680043.8.498.67859043925597585921591407089453470215","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12950639892345812884896649629409199101","series":"1.2.826.0.1.3680043.8.498.81590976414734528793855904341814537604","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111","series":"1.2.826.0.1.3680043.8.498.11744743084974014118553088601233168009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12965787822216495014894819598440431626","series":"1.2.826.0.1.3680043.8.498.56500634209564360805887544020387868566","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12970315889871740202726744767179863179","series":"1.2.826.0.1.3680043.8.498.10566025012824041262283649021269090405","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12980448935316870673341494036873313299","series":"1.2.826.0.1.3680043.8.498.25121430353021414639747047958789120952","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12982826264059547618124186257414818172","series":"1.2.826.0.1.3680043.8.498.12052185743359183142491416639313689947","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12986028147682493711912242081052787346","series":"1.2.826.0.1.3680043.8.498.26978411166954205531746518997831862832","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12992013816638129896738209015383985569","series":"1.2.826.0.1.3680043.8.498.30573706521708061684412855967098269116","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13004048880335015585285100185910148808","series":"1.2.826.0.1.3680043.8.498.43914746234594309439859115600161347132","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13016684730504329799460698013153308088","series":"1.2.826.0.1.3680043.8.498.91102874528160247996778677223428836092","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13022266764623819477228425331705640702","series":"1.2.826.0.1.3680043.8.498.15932372268319292676820979718083943579","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13024625496981667037628275235696732200","series":"1.2.826.0.1.3680043.8.498.12628834490009567208739302636201422513","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13026463185033527579856819607371375594","series":"1.2.826.0.1.3680043.8.498.16879760031666489941348713092899234037","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13027502247317650159940595684214701589","series":"1.2.826.0.1.3680043.8.498.46127653962961269010827143317373397777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229","series":"1.2.826.0.1.3680043.8.498.62256727612139688251154001572262469660","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13042847643086819857036458569632034999","series":"1.2.826.0.1.3680043.8.498.88024536087331137445546377833764880721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13062978180647549151946243290102173508","series":"1.2.826.0.1.3680043.8.498.51656326812342971988994411403519974207","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13069081594164738319108129224400751974","series":"1.2.826.0.1.3680043.8.498.71667334859637043377358788350339098624","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13073387272228394257811861905056479438","series":"1.2.826.0.1.3680043.8.498.60893370578618234818013890971652440165","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703","series":"1.2.826.0.1.3680043.8.498.65295552492535145165970617073308542684","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13079170182386765602129976624023222742","series":"1.2.826.0.1.3680043.8.498.26209157340191525595555502144025012961","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13087770868099843948960419968576882974","series":"1.2.826.0.1.3680043.8.498.71186539001072945874100044017211175600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13093528938282977263747792690249036456","series":"1.2.826.0.1.3680043.8.498.10555838985689977634474865532382581419","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987","series":"1.2.826.0.1.3680043.8.498.79864978602552541511404752012649787731","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13103513835731355705707295729643672786","series":"1.2.826.0.1.3680043.8.498.36649677426293179863821419018740577248","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13113623966675790946816351311283701141","series":"1.2.826.0.1.3680043.8.498.12196534403245116397293824221263360378","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127","series":"1.2.826.0.1.3680043.8.498.89897475511419425442960297650255373981","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13118532243906060645114391749986421707","series":"1.2.826.0.1.3680043.8.498.11753929464309833718139086588914171756","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13126380916005729420163981055214890048","series":"1.2.826.0.1.3680043.8.498.62124233639593824758033776632327189077","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13128506994126383651503283021834372142","series":"1.2.826.0.1.3680043.8.498.40762879428959424460519998984522260487","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13130534736322715018024786213131944607","series":"1.2.826.0.1.3680043.8.498.12248874840318611320070800123390080196","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032","series":"1.2.826.0.1.3680043.8.498.47490086363407024895108797953797574176","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13136392756016978765204420377500607545","series":"1.2.826.0.1.3680043.8.498.22406233191442169744487108472292176140","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13137549927896953291645459480590055925","series":"1.2.826.0.1.3680043.8.498.95941217263098579502456561765065955187","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13146466304138609448483938844487385775","series":"1.2.826.0.1.3680043.8.498.13265240734246373726878306945195045585","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13148805278538458401210722493048566510","series":"1.2.826.0.1.3680043.8.498.12633451350447425935988057405787218958","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13151845892175420505816340845155354956","series":"1.2.826.0.1.3680043.8.498.11534606965451903759247535596671545163","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13155572156594271049520969168607160312","series":"1.2.826.0.1.3680043.8.498.11010836760325649087593876758273433991","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032","series":"1.2.826.0.1.3680043.8.498.10959748074703653226587682023999169285","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13162479706670045075896993021883562680","series":"1.2.826.0.1.3680043.8.498.10031016677302738528133519623700036745","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13165770394160595162984722917452661205","series":"1.2.826.0.1.3680043.8.498.91751989726601023110193739741567273354","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13172295193446466338347273836250944090","series":"1.2.826.0.1.3680043.8.498.11349304102117647036731531173344315180","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173926977743814814616127342102020710","series":"1.2.826.0.1.3680043.8.498.86434338464926043544856735437851674613","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13179169600887013946657470830682778717","series":"1.2.826.0.1.3680043.8.498.32826629687404447834994516959490350712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13199635020145400856987271814458645896","series":"1.2.826.0.1.3680043.8.498.72333966867713249024279617834259396317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495","series":"1.2.826.0.1.3680043.8.498.12118741478214766284539566795612575843","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434","series":"1.2.826.0.1.3680043.8.498.12140588861074331992756277420071897463","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13222515831888962408632292783773537312","series":"1.2.826.0.1.3680043.8.498.12441029855866660632332223255080566640","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13226961394403558608712498872213808226","series":"1.2.826.0.1.3680043.8.498.63367571117738819145506989786185451818","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13232501202175905676335982676386953813","series":"1.2.826.0.1.3680043.8.498.11613676766489417256977430554190698003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308","series":"1.2.826.0.1.3680043.8.498.76895999304274125546345657770099365334","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13246496699722524712272293738359002501","series":"1.2.826.0.1.3680043.8.498.42710485567519166667618072850701868646","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025","series":"1.2.826.0.1.3680043.8.498.26380698986922744062960392679915068007","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13262583253812134425581922368942846684","series":"1.2.826.0.1.3680043.8.498.12073488699793416503424867112402691454","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891","series":"1.2.826.0.1.3680043.8.498.63041321193340816564494138555216466576","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13280017707177438711374726939683941320","series":"1.2.826.0.1.3680043.8.498.59487944185402989245649100487418829238","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13284123714012951608998434802430211634","series":"1.2.826.0.1.3680043.8.498.30201977071757871497896552898870078863","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13288244567806030859347115222642155186","series":"1.2.826.0.1.3680043.8.498.42360166116406021638543810830600580104","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314","series":"1.2.826.0.1.3680043.8.498.57948102594525256813062340005585713742","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13297973188397662905314777604089528518","series":"1.2.826.0.1.3680043.8.498.87414588343784914417212011848281585458","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","series":"1.2.826.0.1.3680043.8.498.72776038008406453005836722041262601605","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13321746459276191365724686301267421531","series":"1.2.826.0.1.3680043.8.498.52397066863307979830101576668347477401","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13330694332190836651711643932953274573","series":"1.2.826.0.1.3680043.8.498.62044640363596939806501296182712744258","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903","series":"1.2.826.0.1.3680043.8.498.44481757782902229277198472340346791285","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13353249891620291438488727876474802879","series":"1.2.826.0.1.3680043.8.498.11334556134041807975217801503874780983","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13353966209257569249620562863006006441","series":"1.2.826.0.1.3680043.8.498.10328923782798955675033399348654701390","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13361877959630120404875545203408927433","series":"1.2.826.0.1.3680043.8.498.72958050841669025886440831827079086384","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13363856251607349805111249441972460575","series":"1.2.826.0.1.3680043.8.498.11305649453613435427270352530719007207","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13368881691140330168876825436602977388","series":"1.2.826.0.1.3680043.8.498.11421308220342411005690795516415285897","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13371001997511542011416510364359588752","series":"1.2.826.0.1.3680043.8.498.11583333594100531919260369520101232171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13383972749798909973472489456126521194","series":"1.2.826.0.1.3680043.8.498.13044196163801397735856706759395619019","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13391735462825209934041441767886593395","series":"1.2.826.0.1.3680043.8.498.13189251439136284751683181074696449187","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13400273795954308944359576841942016981","series":"1.2.826.0.1.3680043.8.498.74057464392700222169612023410587041781","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965","series":"1.2.826.0.1.3680043.8.498.90126907772602711483050382008266323129","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13455212888114594010716499984611216646","series":"1.2.826.0.1.3680043.8.498.96757050548145816683859965530887219391","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13460268079835549295338279322112693785","series":"1.2.826.0.1.3680043.8.498.10215254483285389434699913098277254104","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13463672277039459152171414002273376185","series":"1.2.826.0.1.3680043.8.498.49098601323390422851145639246753988536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13597463452741791628023738080583139636","series":"1.2.826.0.1.3680043.8.498.87300841432857205242441676532234532402","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13660439785395658826946769630921591235","series":"1.2.826.0.1.3680043.8.498.91837780147919383995470616215531865235","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13737588520456064856233576393747389054","series":"1.2.826.0.1.3680043.8.498.66595746739641158724249252174197771162","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13843368750899904654034226115826130084","series":"1.2.826.0.1.3680043.8.498.12280187942957054853995700262870394015","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13958765761660666865188384031150827763","series":"1.2.826.0.1.3680043.8.498.42871905798308533445649781914804574082","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13975027824128345697756313869165838055","series":"1.2.826.0.1.3680043.8.498.34358821941374001943403745300162089888","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14061227335828384505403558788548348446","series":"1.2.826.0.1.3680043.8.498.28575127799946472291432427963369671768","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14117402902169664329719927863268363596","series":"1.2.826.0.1.3680043.8.498.56294445305460122461909800747968453671","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14218113768651197803980865495163189825","series":"1.2.826.0.1.3680043.8.498.17149658253978260117398739129153780168","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14236519693915284584208989450687591759","series":"1.2.826.0.1.3680043.8.498.17250642897655378744080226824913570039","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14325424371811230497993759959862738838","series":"1.2.826.0.1.3680043.8.498.77575566010160706556329046683513958251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14400292895187181989838122627394761532","series":"1.2.826.0.1.3680043.8.498.33995226233721435926603010340724313588","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14472261871417480346162193127520918127","series":"1.2.826.0.1.3680043.8.498.11376639756575546751520075797828416094","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14479913217228604289584726096114526268","series":"1.2.826.0.1.3680043.8.498.10223200602511958865858437348031321954","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14511862461248267537051973802039959211","series":"1.2.826.0.1.3680043.8.498.12776317144904769408299851103772945558","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14728068688166107022558957899468489735","series":"1.2.826.0.1.3680043.8.498.94188021498182416583578842545701822550","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14770617961376953596780345652892307570","series":"1.2.826.0.1.3680043.8.498.10094597128190108337931096851176172480","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14845527334849156639306067473358712770","series":"1.2.826.0.1.3680043.8.498.19526305269452461239491303794781777799","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14958560978176121375035839707004667832","series":"1.2.826.0.1.3680043.8.498.84641347381095464366264671891414034427","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14963472809950919852176661973255059069","series":"1.2.826.0.1.3680043.8.498.55845631348480691429073048531514802875","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15025957400098336605043184930980421418","series":"1.2.826.0.1.3680043.8.498.55119271123212202820221640749400352065","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15090991392324602559982474708414427621","series":"1.2.826.0.1.3680043.8.498.70154063977187292401100334347539112807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15219410096632334470101257478715035071","series":"1.2.826.0.1.3680043.8.498.51871679587091401199635535309418306001","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15297444019822621504849851092382560980","series":"1.2.826.0.1.3680043.8.498.79128882588017263807200318335646818226","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15360150088598749206650025637914821738","series":"1.2.826.0.1.3680043.8.498.95598720853886926259051375667170114612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885","series":"1.2.826.0.1.3680043.8.498.37255372840481417789586340099838684714","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15468179600399954478612037548577772269","series":"1.2.826.0.1.3680043.8.498.51763302722812392498474710603915033209","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15522600887743817464976568872439594397","series":"1.2.826.0.1.3680043.8.498.46080801457795940469406598064384209820","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309","series":"1.2.826.0.1.3680043.8.498.12836209270253860624540818250354794761","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15630146025588580188363070220779173742","series":"1.2.826.0.1.3680043.8.498.11901515020193471383757492965207397930","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840","series":"1.2.826.0.1.3680043.8.498.76182665159350749354843871769685251589","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15969838281154273030237391394632849764","series":"1.2.826.0.1.3680043.8.498.95532946470539933997350249680236813655","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192","series":"1.2.826.0.1.3680043.8.498.10800916297806196226204914061473420340","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16158812105029662172517682642229718094","series":"1.2.826.0.1.3680043.8.498.76937743191974291516102207534993445968","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16189833935983604051983090678102023832","series":"1.2.826.0.1.3680043.8.498.78075198920291459358291861510652705867","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16258190393569006141285550986533848849","series":"1.2.826.0.1.3680043.8.498.13390506611005683476748645884843212805","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16338296537380784459604897543069625390","series":"1.2.826.0.1.3680043.8.498.95021525995965015382128943121265708074","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16397020531009156965821242117417547331","series":"1.2.826.0.1.3680043.8.498.38662665861825841519249648166034217849","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16483470188406187000599644150600279596","series":"1.2.826.0.1.3680043.8.498.22897286622904227722680581663007665880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16538738179840493095338324009830673843","series":"1.2.826.0.1.3680043.8.498.81926917791149376459083707390390998008","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16581426215518472805200227992550139190","series":"1.2.826.0.1.3680043.8.498.56578363306722612823678427868765433458","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849","series":"1.2.826.0.1.3680043.8.498.17262462761589987834957620507973347685","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16709565360569239280032743619008187278","series":"1.2.826.0.1.3680043.8.498.53269817174289964624178019954928294840","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16718788321951483456195507236392554704","series":"1.2.826.0.1.3680043.8.498.28533912546790354644340316035046338844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16812524557513419866531695739014681414","series":"1.2.826.0.1.3680043.8.498.15615904519288998367423966302608190516","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16862596867845295946732698348903803700","series":"1.2.826.0.1.3680043.8.498.29099032648532802328664594335557912748","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16886658450293145802897654768506439494","series":"1.2.826.0.1.3680043.8.498.68120074886840050845716533146421727044","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17056997047488992455301411194938263817","series":"1.2.826.0.1.3680043.8.498.21657113338682682494573349498787597548","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17092147498062993594620309531155267119","series":"1.2.826.0.1.3680043.8.498.53713765237196658026322945543142111312","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17107696939614797679012793547913769707","series":"1.2.826.0.1.3680043.8.498.11588956579551990766989965769026919681","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17160610480070701553118850947260201608","series":"1.2.826.0.1.3680043.8.498.44675071912816975922051773102851117670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17265093113586420061761042467476063876","series":"1.2.826.0.1.3680043.8.498.76127206873042321075993883548429763706","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17336223563862025807066700226971845206","series":"1.2.826.0.1.3680043.8.498.11156648555358624664909376565866843224","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17420673628956367425739018933968443620","series":"1.2.826.0.1.3680043.8.498.60848848114826121529880691160420803730","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17517950899674150719043097786728121608","series":"1.2.826.0.1.3680043.8.498.10318025045146320773196757695566786442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17618734998753079190687719289815512319","series":"1.2.826.0.1.3680043.8.498.60002884547662090302853972307444111328","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17663005209256897892831319144686413853","series":"1.2.826.0.1.3680043.8.498.84216940881041294908338306330925043840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17768582402712001917772688462080201650","series":"1.2.826.0.1.3680043.8.498.19404812819230864885303098706606274688","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17813878802252660525798053577039470087","series":"1.2.826.0.1.3680043.8.498.11067351060599717646765478308674261078","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17856297841306582866782295284505406916","series":"1.2.826.0.1.3680043.8.498.13081920998212819949986092350869297017","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17909590711893200514107254988453660857","series":"1.2.826.0.1.3680043.8.498.11809157981386656760582818441822440163","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730","series":"1.2.826.0.1.3680043.8.498.29721590119604292053685051660598906942","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18090147030578547920423840630147682826","series":"1.2.826.0.1.3680043.8.498.95098166546472630859998119688657891250","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285","series":"1.2.826.0.1.3680043.8.498.12192657819525103266131561003924796391","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18176586296602067352845573876750324761","series":"1.2.826.0.1.3680043.8.498.23609442128442653511576110789373968865","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18245500498843512203508664874652003124","series":"1.2.826.0.1.3680043.8.498.24613471863930037472192356988536115822","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569","series":"1.2.826.0.1.3680043.8.498.77894591706974627283467527327745950493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378","series":"1.2.826.0.1.3680043.8.498.99983693114250264488966342808044947409","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18509793799833391995987656714798904411","series":"1.2.826.0.1.3680043.8.498.12906474775392796890047874825938131032","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18592535981451766048952107991674227570","series":"1.2.826.0.1.3680043.8.498.45829776133686333481972524256595365750","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18625377919309205857120774497713011583","series":"1.2.826.0.1.3680043.8.498.54689841097404830358618014382962870719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18680059761479598675335345641208680006","series":"1.2.826.0.1.3680043.8.498.12213397852325686652861609846521910837","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18721466442792788108431096912844719622","series":"1.2.826.0.1.3680043.8.498.14198863253571655528615586887543292647","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18792568294433896490648198192877693850","series":"1.2.826.0.1.3680043.8.498.10904204697016146307977151581212806887","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18825393854436144614202928260751418496","series":"1.2.826.0.1.3680043.8.498.61167626063546758010972524508257646087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18899625751876258520147203904985857313","series":"1.2.826.0.1.3680043.8.498.11410436636075436127432390675834014377","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18993960105596277135383396226525760874","series":"1.2.826.0.1.3680043.8.498.25297183727980697448110886491285409661","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.19104350033985745058396432133828748480","series":"1.2.826.0.1.3680043.8.498.87792789550747907071280726737208338592","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19178214677492694135853596920291058397","series":"1.2.826.0.1.3680043.8.498.11699190701474131419644117844472408321","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19197037365998103318720742154440921358","series":"1.2.826.0.1.3680043.8.498.33340886658897049090745765854195773116","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19291375342005405073321673638971145777","series":"1.2.826.0.1.3680043.8.498.30957353422002248521481632673974913924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19419878169415950073207624537163068568","series":"1.2.826.0.1.3680043.8.498.41806407617390043162776963261493180977","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19489875090732464985703624680756212359","series":"1.2.826.0.1.3680043.8.498.14614427991097310846992643185516766495","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19516545000256238042886469152307943182","series":"1.2.826.0.1.3680043.8.498.83894398980609129965979863390598300709","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19601428113073026637547887698342670296","series":"1.2.826.0.1.3680043.8.498.49534779674331653989909789163950049263","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19706775737718777290660344285046025310","series":"1.2.826.0.1.3680043.8.498.40988113481459655340517857363130484226","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955","series":"1.2.826.0.1.3680043.8.498.46450623079341377184282090353547333553","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19830663206351645385520268848711462246","series":"1.2.826.0.1.3680043.8.498.43239088394215642405340338158380018670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760","series":"1.2.826.0.1.3680043.8.498.61479155371069084314682825031157959116","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19895710326297085463251295629948896948","series":"1.2.826.0.1.3680043.8.498.67542922061718156841895421593032558081","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19965044457637713843792494926652048462","series":"1.2.826.0.1.3680043.8.498.90586230071070700171633293127472610003","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20063306448698185569553477478719533390","series":"1.2.826.0.1.3680043.8.498.12957807429037643884499493892596548170","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20156361590688072955579948030116579099","series":"1.2.826.0.1.3680043.8.498.78333421841381892498110533815117739362","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20192024727432864451309151144591371229","series":"1.2.826.0.1.3680043.8.498.30805291642060106124242691031472812791","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20232953944759707220316171133595554284","series":"1.2.826.0.1.3680043.8.498.86545704130244499050147753600066095908","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20301796413163444642416918108994411350","series":"1.2.826.0.1.3680043.8.498.12827367693964151321854236507733665805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20411068889491114730824265411248922690","series":"1.2.826.0.1.3680043.8.498.65365698403419319655554924214469093492","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007","series":"1.2.826.0.1.3680043.8.498.32929684725295619149645060101758871526","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","series":"1.2.826.0.1.3680043.8.498.42760874564795213002712682638757042555","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20587514285427671071934631922521946816","series":"1.2.826.0.1.3680043.8.498.11548961287201889899177170065607678332","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785","series":"1.2.826.0.1.3680043.8.498.42424897503694901917876578829512225714","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784","series":"1.2.826.0.1.3680043.8.498.41067340495720672970169666334930835567","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20850785960533583051456342090773230767","series":"1.2.826.0.1.3680043.8.498.11828838034955874542942898062091784458","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20933151653103548000061448393771614270","series":"1.2.826.0.1.3680043.8.498.12776340895980971364222014413708932710","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20953536352252551462901259297974956430","series":"1.2.826.0.1.3680043.8.498.32688234939762400252660442108317529666","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954","series":"1.2.826.0.1.3680043.8.498.33362799521324252504525584889432002824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21116046804798247223618072736212944615","series":"1.2.826.0.1.3680043.8.498.11295995813048917857897008106585278976","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21239630702461934662855021164119114203","series":"1.2.826.0.1.3680043.8.498.50460137053640795246174653131566318968","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21312254534068992054745674608909722454","series":"1.2.826.0.1.3680043.8.498.70519981279366287029477659588116463346","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21389336408905524316915532623385847283","series":"1.2.826.0.1.3680043.8.498.80351487496445035957021274802393258166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21449012857520712901884874638467532958","series":"1.2.826.0.1.3680043.8.498.46868322819853201327476159796033770128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21520719119040558609068952147312604007","series":"1.2.826.0.1.3680043.8.498.81386786135438175200303028554043057252","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21584859831565112159503971937497384030","series":"1.2.826.0.1.3680043.8.498.58248619180494662361876037929600896223","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21634792491972658759276025394685307053","series":"1.2.826.0.1.3680043.8.498.12138334607133313196861487573264434447","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21726746202429506433532477857404854137","series":"1.2.826.0.1.3680043.8.498.95292647411766036485004743977409790637","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21862609534273524003153225756948913437","series":"1.2.826.0.1.3680043.8.498.39745239140480014367975734241258609521","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673","series":"1.2.826.0.1.3680043.8.498.99268646549647053641058841619189449843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21954482915702797371609582020715955762","series":"1.2.826.0.1.3680043.8.498.67187919244043573230039792230338280764","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22007253197523827371600763959266047295","series":"1.2.826.0.1.3680043.8.498.41595218522393674579280600324177695128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22027914323057444746912555313058666950","series":"1.2.826.0.1.3680043.8.498.43050587631222553896410266203385800075","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22088734384869383587217556574382319654","series":"1.2.826.0.1.3680043.8.498.69638440217878765278927666552270459762","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22119720417177816110156744770171496965","series":"1.2.826.0.1.3680043.8.498.11518347023788527266354862352010423284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22137222939487357840166329770642627189","series":"1.2.826.0.1.3680043.8.498.90596116177013154202343640372048866295","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22263857526032448247006468310072414129","series":"1.2.826.0.1.3680043.8.498.12467908070990617384352154517671611120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22357754074308826479280023393343731000","series":"1.2.826.0.1.3680043.8.498.64509351439204561124205676003142261196","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22398731807962276404889164827238996005","series":"1.2.826.0.1.3680043.8.498.64553204054720700079618351898121953392","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22426390571898968732573184721547490379","series":"1.2.826.0.1.3680043.8.498.20034393726740695827868628178375684459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22508124641661887430614105328316053644","series":"1.2.826.0.1.3680043.8.498.24029597321660783889021282538711798618","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22594249996322566888700984133307337128","series":"1.2.826.0.1.3680043.8.498.62502654870614525659224600863152472360","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22617494262793136174349268379191631799","series":"1.2.826.0.1.3680043.8.498.45162131665018136819829223324170262840","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22656222254273662937226093008714426490","series":"1.2.826.0.1.3680043.8.498.33563144347046850616229494586496407515","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22803020644232321685575897661362804423","series":"1.2.826.0.1.3680043.8.498.12423083258874298696639022964496927018","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22938598188283919466040132624303635817","series":"1.2.826.0.1.3680043.8.498.10486878882150934563306323674245213448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22990786875064680564672188701015740170","series":"1.2.826.0.1.3680043.8.498.59292102789921260567676269047447129077","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23046618786812839762801821364158313134","series":"1.2.826.0.1.3680043.8.498.61253592500888755458885642191150711706","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23073694267101759922738121732233014980","series":"1.2.826.0.1.3680043.8.498.11977193893960515984685479353604337831","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23094733099955217022059584418468076048","series":"1.2.826.0.1.3680043.8.498.36296029079957551969013423439568103249","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23182005205627494484490011351673355568","series":"1.2.826.0.1.3680043.8.498.90085448057739273390636382795903619227","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932","series":"1.2.826.0.1.3680043.8.498.99016392436099887338494530286934459754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23427198594727947883953289189443006288","series":"1.2.826.0.1.3680043.8.498.11409597031553463185211699463755861920","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663","series":"1.2.826.0.1.3680043.8.498.61243156715172118200610876010365834100","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23713320796530316370039202981138376061","series":"1.2.826.0.1.3680043.8.498.13231620877753887462181456188782677855","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23733147518259581649562651915997449088","series":"1.2.826.0.1.3680043.8.498.10226041413486864691002886667352603128","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23833933822461809237193998300076847210","series":"1.2.826.0.1.3680043.8.498.11247052448044063118394861370173314336","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.23939313916974849526459717576344063277","series":"1.2.826.0.1.3680043.8.498.48980657576906720050085996149170059791","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23986051274386630509877205174270047762","series":"1.2.826.0.1.3680043.8.498.11889274897805361052463444823097842994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24101792822866991033671006059283560916","series":"1.2.826.0.1.3680043.8.498.35178892330312526025242964697591071755","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24209643225939781032698894992032106781","series":"1.2.826.0.1.3680043.8.498.71434245025293911169161210286156018615","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24267153760943721146187182369810590631","series":"1.2.826.0.1.3680043.8.498.49672275578894090248657860384611525673","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24291465113360632954006568021942978765","series":"1.2.826.0.1.3680043.8.498.10497422130750866359443026559314056807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24369332907731805266450307536793220148","series":"1.2.826.0.1.3680043.8.498.11165252464726566589184956532566907267","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24478195415117310702720951368429189290","series":"1.2.826.0.1.3680043.8.498.40944868985820883186888578097766259045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24555528844026121007453886375651403314","series":"1.2.826.0.1.3680043.8.498.11549213876970033121794064638899639384","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24635589616467409099788838058892886889","series":"1.2.826.0.1.3680043.8.498.75687329675914753494029747161612812601","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24679905857546029995247839380830345841","series":"1.2.826.0.1.3680043.8.498.12467323665997321939143587665965898403","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038","series":"1.2.826.0.1.3680043.8.498.12754040892284764694413359852388488118","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24826041178550854190310664677274404684","series":"1.2.826.0.1.3680043.8.498.83325981508119308742242504805976381421","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24875052319550288230112434151792125187","series":"1.2.826.0.1.3680043.8.498.70048579514557033690451756448195142755","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25006652485159423287561656594171884310","series":"1.2.826.0.1.3680043.8.498.76574341967798447024520900823859772087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25103708168569906591706406794385334861","series":"1.2.826.0.1.3680043.8.498.23371056365841634080545878979382572737","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047","series":"1.2.826.0.1.3680043.8.498.94145392694842484435104970385732489765","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25188196199869378521949969245728443004","series":"1.2.826.0.1.3680043.8.498.91410822992028809945361353198581154193","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25206096958520407302630747466352252249","series":"1.2.826.0.1.3680043.8.498.61784024144215772647451629850762732203","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25266231062774974303723359806535863810","series":"1.2.826.0.1.3680043.8.498.20453109350314436866483740937175879232","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25359770839263995354733835365510047109","series":"1.2.826.0.1.3680043.8.498.12196060816487505855282810661300363466","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25440342930186170531037102906687829898","series":"1.2.826.0.1.3680043.8.498.13126797635568389469871381358035871355","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25487225187305802150215261512092051199","series":"1.2.826.0.1.3680043.8.498.30546827103810831612846325783766654087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25549974159486441938843396591725289882","series":"1.2.826.0.1.3680043.8.498.39931841223499963056910631071107808746","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25558526735614916746398919603493693652","series":"1.2.826.0.1.3680043.8.498.35782351866367011994814907245015720951","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25635522141852534580088120697977308505","series":"1.2.826.0.1.3680043.8.498.11525273190495616886704798039407254672","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25650384476012633612528314070894186451","series":"1.2.826.0.1.3680043.8.498.58079973226727116191718249042410618176","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487","series":"1.2.826.0.1.3680043.8.498.34260834550457679073635287817513014352","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25697539332996211770844878649053316629","series":"1.2.826.0.1.3680043.8.498.10802207115635338997296234014430264794","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25766387275525303860953321689718246405","series":"1.2.826.0.1.3680043.8.498.12360771775091423809183536967950506892","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25792216289349936450377008453709520260","series":"1.2.826.0.1.3680043.8.498.13170496526512194260125355005169783557","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25856396695552621816673203639618612020","series":"1.2.826.0.1.3680043.8.498.87473802545515033824731235073403371853","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25904473139226718487560978103077541725","series":"1.2.826.0.1.3680043.8.498.45740250117147783530378791195234328020","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036","series":"1.2.826.0.1.3680043.8.498.81383762313594490464030019117181047266","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26002972602557466222471260313257965551","series":"1.2.826.0.1.3680043.8.498.84676258201798022827952352407242236870","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26023120405608635089544039694812003604","series":"1.2.826.0.1.3680043.8.498.10291061323658794309113805318110098227","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26072622476167889001255150030772906081","series":"1.2.826.0.1.3680043.8.498.80178119670542339882399667981171343759","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26257425287121280175418060407044075705","series":"1.2.826.0.1.3680043.8.498.10038308113249402179891549746533150089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26325951519908477683595799515094966069","series":"1.2.826.0.1.3680043.8.498.11594962606190230857651212272662704943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730","series":"1.2.826.0.1.3680043.8.498.12510392199893595332079004548286376584","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669","series":"1.2.826.0.1.3680043.8.498.28761268253720255800239982364106618996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26512333939158074470511885252122191522","series":"1.2.826.0.1.3680043.8.498.43309588032822734826435523894698569610","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26542435427465415008860869796504112727","series":"1.2.826.0.1.3680043.8.498.57928617754493153356603211876634113625","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26698126402727061400851465377574886112","series":"1.2.826.0.1.3680043.8.498.13373872543997584166033689392932893516","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26792665054823243997838322477587733719","series":"1.2.826.0.1.3680043.8.498.94266101128661337865918060280767634351","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26843088620128166809417361066548447201","series":"1.2.826.0.1.3680043.8.498.23429532435923317882737919863393787881","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26880562799522930458745136318439622601","series":"1.2.826.0.1.3680043.8.498.86691612756565808808636503474651517998","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26978027400525885904983020900996228004","series":"1.2.826.0.1.3680043.8.498.11744911536645495385787524557440429934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27023892731021470476500135946317920771","series":"1.2.826.0.1.3680043.8.498.43346636675574017731147858565145038986","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27059975738703546675398620281849285863","series":"1.2.826.0.1.3680043.8.498.52628856410485849404743757714899894398","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246","series":"1.2.826.0.1.3680043.8.498.78700444344419223048745612092665344515","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27191574009501966309382307682020135126","series":"1.2.826.0.1.3680043.8.498.18758737720996040069611412597233405760","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27197413293500684424735285039923168926","series":"1.2.826.0.1.3680043.8.498.12523453025354446805230040182405969866","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545","series":"1.2.826.0.1.3680043.8.498.10586452302548921493053899923883225006","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27311413427137994435573903541974122355","series":"1.2.826.0.1.3680043.8.498.60991584585177540090554423796179541819","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27339999385861467858422021400023533486","series":"1.2.826.0.1.3680043.8.498.80945306786763636733257247702623730056","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27422742748182689233651649793019570511","series":"1.2.826.0.1.3680043.8.498.19604596287847967601601839723311301990","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27456622705361831116491505615162380604","series":"1.2.826.0.1.3680043.8.498.93661741281795489071056220918963321561","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27593710915588175306316595263625481601","series":"1.2.826.0.1.3680043.8.498.35148501199753016219693427110300815822","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27644321991318957631114727895883331622","series":"1.2.826.0.1.3680043.8.498.32060740759650718493593169408111799758","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27656830051511853680197276412365020780","series":"1.2.826.0.1.3680043.8.498.62758576861424401678467581630554502162","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924","series":"1.2.826.0.1.3680043.8.498.10430108675976725353539977125906268009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721","series":"1.2.826.0.1.3680043.8.498.10421323280640142846460024157322746944","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27793455627426963285485957211105588905","series":"1.2.826.0.1.3680043.8.498.65272160966177663431448811044158785133","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27883371281368994280894952749865160316","series":"1.2.826.0.1.3680043.8.498.95444919594670872179534383673664405156","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27921111050680927418542588425335218896","series":"1.2.826.0.1.3680043.8.498.16126717924259050409338714730334553370","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27946605663086243142356841956045514227","series":"1.2.826.0.1.3680043.8.498.59286132867057790560740967962989151466","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28011057282528460306257420499560926354","series":"1.2.826.0.1.3680043.8.498.78390745603902420523425491127223886659","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28040645186226259590182363324146810031","series":"1.2.826.0.1.3680043.8.498.11885480950723007271245615893490522335","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28132506470634542229651932141193521661","series":"1.2.826.0.1.3680043.8.498.55630768274297148095017472869757455369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28218476802410349760103243990109474475","series":"1.2.826.0.1.3680043.8.498.30079231528904836540382944698853050566","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","series":"1.2.826.0.1.3680043.8.498.32033187650285484285843363692269589066","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28295265939946677460838974253299313798","series":"1.2.826.0.1.3680043.8.498.81097576789330421064345514856550754480","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28333127304382930639614800697959296306","series":"1.2.826.0.1.3680043.8.498.72586938088967649583073916587783104842","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28425280718252632810092859612928047098","series":"1.2.826.0.1.3680043.8.498.86030729065495451292405195079827907197","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948","series":"1.2.826.0.1.3680043.8.498.56121508730642526082032872713806566229","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28539130184243787339297053654373783542","series":"1.2.826.0.1.3680043.8.498.10772351819617631644032859553774294363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28626044336302232656987963623687293705","series":"1.2.826.0.1.3680043.8.498.82677537152732364810610717397132750054","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719","series":"1.2.826.0.1.3680043.8.498.11951256066396367612526583542678222651","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28646304373057947277669139986982208473","series":"1.2.826.0.1.3680043.8.498.97830809997841763552096385114268940649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28686498221359515429184385385077600687","series":"1.2.826.0.1.3680043.8.498.94850441761448309745632361155034448983","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28770594655815108010532953376271504596","series":"1.2.826.0.1.3680043.8.498.70028056887469304606592555777642734664","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28848511016500781798837621261660844237","series":"1.2.826.0.1.3680043.8.498.94688407362016678917453183135435406606","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608","series":"1.2.826.0.1.3680043.8.498.48669435010734548926088990885351646056","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28946722216615442736212359468410891872","series":"1.2.826.0.1.3680043.8.498.11940174894976771009036785310575293105","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29039872266866623711060910233238440882","series":"1.2.826.0.1.3680043.8.498.12869397676148451644017774280524061206","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29227028592609694700827329867343028944","series":"1.2.826.0.1.3680043.8.498.48661874730175704216729182026702999110","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29292398668403593100514493322849770181","series":"1.2.826.0.1.3680043.8.498.70910712265247387623888775767971449501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29354469534263517460113311852821643665","series":"1.2.826.0.1.3680043.8.498.24169729653905315031484394146967050006","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29371692234061436334815897970798625595","series":"1.2.826.0.1.3680043.8.498.10523138176118007028926703179748649379","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29384016102624554484300773136383156153","series":"1.2.826.0.1.3680043.8.498.46419368822169352673062925700448075522","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29473278891551292295080887778714088878","series":"1.2.826.0.1.3680043.8.498.60469371069242846744039037659375675190","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29579560656789178333629609998252952856","series":"1.2.826.0.1.3680043.8.498.11463466628492424245314612665650892748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29741673719687535294395668150749894623","series":"1.2.826.0.1.3680043.8.498.36383415101972449797456537189912291291","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550","series":"1.2.826.0.1.3680043.8.498.77755509181820937582475082973876194850","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29866827083314645739444078187995496049","series":"1.2.826.0.1.3680043.8.498.12247836491947814384365993916361852785","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29976611818313906970687136815346573471","series":"1.2.826.0.1.3680043.8.498.38569343395735479723500180286354266822","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30066410959464367576375502522994033142","series":"1.2.826.0.1.3680043.8.498.65157118153209645671334394023955536238","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922","series":"1.2.826.0.1.3680043.8.498.18893444986254332630190214700602412073","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30174064884443577177838026250637902388","series":"1.2.826.0.1.3680043.8.498.12829703652044011094257295539659204933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30222189977108980260768952691472020722","series":"1.2.826.0.1.3680043.8.498.31484617641218130862047660690246746083","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30248682294786626375369374249874501600","series":"1.2.826.0.1.3680043.8.498.29110460382510589832788877561974664771","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30264599677934095429161619981918944098","series":"1.2.826.0.1.3680043.8.498.11919543489779934034792762783578444081","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30381005430261460472880465705496199540","series":"1.2.826.0.1.3680043.8.498.12533805743241269273387634265472321388","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30423643741419627789701272644909134762","series":"1.2.826.0.1.3680043.8.498.27556438001949963501805247955142999375","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989","series":"1.2.826.0.1.3680043.8.498.99726227959669579853087903848318649145","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30537361712821464927469618193893850465","series":"1.2.826.0.1.3680043.8.498.13261139833909603392198041066472313667","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30613528335797277620843542280889823612","series":"1.2.826.0.1.3680043.8.498.25401436056144677958958705238338517817","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30673411238679998084995204090753297949","series":"1.2.826.0.1.3680043.8.498.20945626950877989652946364992919030839","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30746167840159843733412534591988948270","series":"1.2.826.0.1.3680043.8.498.13074289174541842976053004241046095101","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579","series":"1.2.826.0.1.3680043.8.498.11530309532651880934013402992495542632","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30836611607735372299752226690388090391","series":"1.2.826.0.1.3680043.8.498.12123370367040527200061728616807213036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30842406081180392627544336691360658070","series":"1.2.826.0.1.3680043.8.498.17288157504346048031452294075896184461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30947630416867960170317316514422933066","series":"1.2.826.0.1.3680043.8.498.12035327983405683701200569383482429674","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30998941785989496591079894962766360976","series":"1.2.826.0.1.3680043.8.498.51480888587969671223855673477958497319","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31080452089421812812019413302222437434","series":"1.2.826.0.1.3680043.8.498.18657547827515406204383520751565437583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31137162827930164367134811210741362416","series":"1.2.826.0.1.3680043.8.498.98089374102748422926522718070878288228","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31175171249406459042393597291141275669","series":"1.2.826.0.1.3680043.8.498.10467565478080414540106809928062339472","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31193772347652083986607183406944630716","series":"1.2.826.0.1.3680043.8.498.12999381162434582070748886784228521250","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","series":"1.2.826.0.1.3680043.8.498.41658661181313939050182618326068905407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31282366761674650660824706600044115805","series":"1.2.826.0.1.3680043.8.498.18535848529936912963313417099912518191","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31352187408311685267245948214616940728","series":"1.2.826.0.1.3680043.8.498.81467903885132559244409926243868915219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31461240229123701082444627629841504099","series":"1.2.826.0.1.3680043.8.498.66978950770926960778429982361093715398","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31565705236407722278921437231164204914","series":"1.2.826.0.1.3680043.8.498.12907384873526046659696886746671720942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31640159014711750808214354395404624815","series":"1.2.826.0.1.3680043.8.498.13241259743717542076295590925847429014","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31792234576444017394182974951507838927","series":"1.2.826.0.1.3680043.8.498.60345571081763895487355818610172879547","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31842732064173292828619477783097732181","series":"1.2.826.0.1.3680043.8.498.11725063359650316870689695043497167137","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31898017986762601111404430885432769100","series":"1.2.826.0.1.3680043.8.498.11349775212217115245975448519302753872","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31949224274318638796196151105617844024","series":"1.2.826.0.1.3680043.8.498.59463140672332282172465816594055336858","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32078592787308498852570411454111970699","series":"1.2.826.0.1.3680043.8.498.14495612480527561989286151063803557747","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32172672595181417103935593872811182816","series":"1.2.826.0.1.3680043.8.498.28029026915035454501271902854630607244","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32241219263926672894433308808653719495","series":"1.2.826.0.1.3680043.8.498.43782284367812669398095103143944928972","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631","series":"1.2.826.0.1.3680043.8.498.13053291722425292133328091523774029451","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32328699307307661181881949763360098439","series":"1.2.826.0.1.3680043.8.498.38717449866400770234154485012758039449","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32417506901165787885027994386927321654","series":"1.2.826.0.1.3680043.8.498.60996082917212496063506930968412982059","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32488933488399304214586978746578986823","series":"1.2.826.0.1.3680043.8.498.10499908214625101255895855623056139758","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32513010438338970418994141548516096399","series":"1.2.826.0.1.3680043.8.498.28168470486944613557779259890952247939","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32571577475600416042179474541176580933","series":"1.2.826.0.1.3680043.8.498.20187026050171240531040218538180218239","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32624291809937483520460431326945616461","series":"1.2.826.0.1.3680043.8.498.26756759643520026517849937163874615115","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32704159300269838731722725240928883385","series":"1.2.826.0.1.3680043.8.498.19410523687045198998847435653347510007","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32784816158214124982843838883515640569","series":"1.2.826.0.1.3680043.8.498.12776705495919349528080027857873963333","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32862727544234948830522036721585095753","series":"1.2.826.0.1.3680043.8.498.34288193138905624994344349962943377429","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32916544979513354644075485240094539251","series":"1.2.826.0.1.3680043.8.498.96090528657202608689704159438865162168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33085434860684110795966817803985794300","series":"1.2.826.0.1.3680043.8.498.50077020120550465675940794420437010495","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071","series":"1.2.826.0.1.3680043.8.498.23209448596594632671652497349254865301","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33273874534046658504935877940537027702","series":"1.2.826.0.1.3680043.8.498.44685250227671431030403471147543658278","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33324670630108603340869986205298694836","series":"1.2.826.0.1.3680043.8.498.11501891878994631725577887114488732421","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33387430823462011959198830568803117633","series":"1.2.826.0.1.3680043.8.498.78719651402767666867050654355158872478","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33427080769740419588952225424281396684","series":"1.2.826.0.1.3680043.8.498.61981342699629821686124780528647413880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550","series":"1.2.826.0.1.3680043.8.498.16870816661357568345887952651674413866","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33620268340279901383196638228030589893","series":"1.2.826.0.1.3680043.8.498.11840608951467094689824816965675636136","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33709759248055631658322056320849494088","series":"1.2.826.0.1.3680043.8.498.53033526532539447175758887520263347169","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33834227140201599591317512187502720859","series":"1.2.826.0.1.3680043.8.498.47732851680653675640939192911836625490","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650","series":"1.2.826.0.1.3680043.8.498.11813658567039665325632827278519844275","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33947148016115721563601115463364787679","series":"1.2.826.0.1.3680043.8.498.98821953962943681539591460653267217068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33997019840409023169542161093833920630","series":"1.2.826.0.1.3680043.8.498.30041006491989554869486102465136266883","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34058295828142876023659425586740832405","series":"1.2.826.0.1.3680043.8.498.10238616576525085120298891402195101756","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34096149829586714035779314398766243514","series":"1.2.826.0.1.3680043.8.498.51436660610445128194653096704783814169","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34206043272866138622475050809785944941","series":"1.2.826.0.1.3680043.8.498.13117684907116608188527245596501676752","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34294522885703132478119231619149118745","series":"1.2.826.0.1.3680043.8.498.11551454000378799855667969590315637616","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34379812274299658721362253287768041613","series":"1.2.826.0.1.3680043.8.498.65916170782720851133927522759538829743","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34441662343660776323013762699640240389","series":"1.2.826.0.1.3680043.8.498.59914735027311303805460444481921321041","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34501976934829436985900782482912019868","series":"1.2.826.0.1.3680043.8.498.10961400602538597588159526475424176016","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34572444983416782410188788741787280444","series":"1.2.826.0.1.3680043.8.498.11359034123799322953387275895267786089","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906","series":"1.2.826.0.1.3680043.8.498.92293494075897916800652855123416612445","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455","series":"1.2.826.0.1.3680043.8.498.75050969850556072169742705802249437589","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599","series":"1.2.826.0.1.3680043.8.498.10697325614403149079670323973202708032","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34791780913322886484171421115861604853","series":"1.2.826.0.1.3680043.8.498.13026261312571984114129508754731641941","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34858782760614383566891001610985452165","series":"1.2.826.0.1.3680043.8.498.83222218376814317942406633577130446687","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579","series":"1.2.826.0.1.3680043.8.498.19153299678437258661595290800072150699","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35017973257585430544005763350601015208","series":"1.2.826.0.1.3680043.8.498.64369352731798895272474479209187555158","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35027750475162395113025853006474122989","series":"1.2.826.0.1.3680043.8.498.78122515320849502665664785222284861988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35121262802959257590413582180924083425","series":"1.2.826.0.1.3680043.8.498.89709281990171897292510962809400999895","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761","series":"1.2.826.0.1.3680043.8.498.19913218922439257245986953710565831795","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35257158249961136373696255071228322165","series":"1.2.826.0.1.3680043.8.498.37062642474787867958991379014412576870","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35338198410583984337908932327787063577","series":"1.2.826.0.1.3680043.8.498.41440011106863747555548415880173969201","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35542843700847404584397532402313561503","series":"1.2.826.0.1.3680043.8.498.43177137609954802554536268858854916050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35633121799529901420381085923067525212","series":"1.2.826.0.1.3680043.8.498.57254828843777130005575024757782416777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35664741425496814074030858561751287211","series":"1.2.826.0.1.3680043.8.498.13051783784828467044497599122552388826","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35769002792415947424087098083674166777","series":"1.2.826.0.1.3680043.8.498.55556652531082123174062330503272879534","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35790589718590759254855277551289842944","series":"1.2.826.0.1.3680043.8.498.18133500415309963361652422523388447708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35826402289277331555736683320175912112","series":"1.2.826.0.1.3680043.8.498.69231989948242049646736334592670523912","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35853404993418331599021266761280794201","series":"1.2.826.0.1.3680043.8.498.45906930536658758912293363502243541288","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314","series":"1.2.826.0.1.3680043.8.498.78558571284417482445005079120947694029","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35923183633813281697521537744486885300","series":"1.2.826.0.1.3680043.8.498.64768537996711607949060065458438390235","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35932368810339127867306228699757809449","series":"1.2.826.0.1.3680043.8.498.72653170478734431337574381948174984945","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35961087303991442751747064990497975939","series":"1.2.826.0.1.3680043.8.498.72539331612088678728973496374344689176","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35991881811942962264382690224767291006","series":"1.2.826.0.1.3680043.8.498.24786035464461391100135409318214306324","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36083794505435625841225237415909011547","series":"1.2.826.0.1.3680043.8.498.82663539423967555414788529366620642037","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36131369236532190338276701919104598475","series":"1.2.826.0.1.3680043.8.498.75125032501215162074160354917205720473","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36185754164795053628650116141620545139","series":"1.2.826.0.1.3680043.8.498.10298882942125093685507773741396498733","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278","series":"1.2.826.0.1.3680043.8.498.33050277184766172534340834160143013972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36314809560682514762544085885005264167","series":"1.2.826.0.1.3680043.8.498.94870438919215068914810660360561995198","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36431753977367847587395456605556703040","series":"1.2.826.0.1.3680043.8.498.84217648499387873850871834353310593519","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36521162836904095186654423648490092035","series":"1.2.826.0.1.3680043.8.498.10161351104437179601692326802555500283","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36599566036331923007586730435849419272","series":"1.2.826.0.1.3680043.8.498.11175521672694273128524214356354676773","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533","series":"1.2.826.0.1.3680043.8.498.89315259688425991847731574035371876500","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36743562653862381001280112332273495985","series":"1.2.826.0.1.3680043.8.498.62151868570145568821981936883081242954","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36812284468308169396147366740331573100","series":"1.2.826.0.1.3680043.8.498.17652697615959989874442589700569069599","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36891467514776207809384804066253854193","series":"1.2.826.0.1.3680043.8.498.92891034343776786256304529409881863177","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36924091633099385100942228071545017565","series":"1.2.826.0.1.3680043.8.498.55262361759564149328973536472322536984","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37005775889857426034942587344620888398","series":"1.2.826.0.1.3680043.8.498.60349073488740723156423969807601572027","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37030593568962187244853961673610373876","series":"1.2.826.0.1.3680043.8.498.12219940439797875460557181898472789227","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727","series":"1.2.826.0.1.3680043.8.498.61073308940448941955788581743502830926","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535","series":"1.2.826.0.1.3680043.8.498.35898040918710096013232196875756583104","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316","series":"1.2.826.0.1.3680043.8.498.13067579367564274301393723267666139529","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37347132459817985668038603215490355489","series":"1.2.826.0.1.3680043.8.498.95989729399502472653472241710824334842","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37390494423735156928337187351900345080","series":"1.2.826.0.1.3680043.8.498.35679516599021040814360595424535159361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37427993203943111265113924891058049338","series":"1.2.826.0.1.3680043.8.498.57464490409972577962382468570658976210","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37438631765770807892914198396852544668","series":"1.2.826.0.1.3680043.8.498.36523088667957457301244679161051099964","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37485075271886447709158414670394650122","series":"1.2.826.0.1.3680043.8.498.10109710642798592293014977999655447032","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37510478969810784660124747242051166554","series":"1.2.826.0.1.3680043.8.498.49423328785128486447962607398630807516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37572710131396388050286115883363501023","series":"1.2.826.0.1.3680043.8.498.12126123445215923593063983473421636550","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903","series":"1.2.826.0.1.3680043.8.498.12490454017065613173843964358392724396","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37644754953609976838298110418046213654","series":"1.2.826.0.1.3680043.8.498.11578810068042119138368537236833484294","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37713008080610827760634302982073393454","series":"1.2.826.0.1.3680043.8.498.39348354594598785665967877938922887925","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37833587429731221455928642963031995680","series":"1.2.826.0.1.3680043.8.498.44371997195817413982467477951770244117","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606","series":"1.2.826.0.1.3680043.8.498.49804878173195931386124464507151514923","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37885316830093262977685802809269466252","series":"1.2.826.0.1.3680043.8.498.47677787456619788464749163762514469630","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37936061471366103885185528435327650981","series":"1.2.826.0.1.3680043.8.498.69394533200478601080819713017086347367","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38010459551726691601986492657296948013","series":"1.2.826.0.1.3680043.8.498.11880880557781591309457994376185552944","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38056024157303576597111203925091929489","series":"1.2.826.0.1.3680043.8.498.59583296032063039783036802461420697474","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38085359363401170553191582038767900323","series":"1.2.826.0.1.3680043.8.498.62917667130015604584216133547373231896","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38173249623713271734596884672770674290","series":"1.2.826.0.1.3680043.8.498.24931563655909426467023482193823000418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38233845434092134418342290639986261345","series":"1.2.826.0.1.3680043.8.498.10308716421492398115622165016146518941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38303182087206935163155565441234078389","series":"1.2.826.0.1.3680043.8.498.11456368913339470162985078890598585192","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38306262614552497897681886368029860358","series":"1.2.826.0.1.3680043.8.498.29410217216623329112194094329343215781","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38361762550587464310603057450403132552","series":"1.2.826.0.1.3680043.8.498.43347405053399899361409220131199180078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38373204312077453422262347769111605003","series":"1.2.826.0.1.3680043.8.498.59164547348228780736735187858309680112","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38427680930662301114332563903957471851","series":"1.2.826.0.1.3680043.8.498.89100287865846404326995509089106708261","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359","series":"1.2.826.0.1.3680043.8.498.12673314975077674643732942183077047192","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267","series":"1.2.826.0.1.3680043.8.498.64713346715310947612954572765808866316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38591915802992557518846862828618375715","series":"1.2.826.0.1.3680043.8.498.46509193904490746075917549807392119696","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38617606118499305781729905815407382451","series":"1.2.826.0.1.3680043.8.498.58748294474703368686796496941523650493","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38655145317957160440692625317823254736","series":"1.2.826.0.1.3680043.8.498.36372399921117677867795693501969578671","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38717329577625255474350472405748992219","series":"1.2.826.0.1.3680043.8.498.66888040920096266668619571690056280400","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38774313460931748278644354956408578813","series":"1.2.826.0.1.3680043.8.498.10420914111239958777587518822615335676","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38844149478205202385909887748058004820","series":"1.2.826.0.1.3680043.8.498.76717884054436998308622331137493746493","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38893522232848525819919092148478879455","series":"1.2.826.0.1.3680043.8.498.93694766669442239605768096672920790130","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38900640030483599614088224967855884221","series":"1.2.826.0.1.3680043.8.498.42952836062620059040420257106613063088","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38928323954764175259962816410233336718","series":"1.2.826.0.1.3680043.8.498.10831971930546414756368379593369752148","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38974023308452365783768081643215200948","series":"1.2.826.0.1.3680043.8.498.20698550692690810140562316754527201109","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354","series":"1.2.826.0.1.3680043.8.498.11167653733538717296577531392729439121","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427","series":"1.2.826.0.1.3680043.8.498.90016982436257865286869111933906843633","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39070169128115370701189151138307035053","series":"1.2.826.0.1.3680043.8.498.95417539995422221553409355320944708295","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39197976722851762826265579540678924584","series":"1.2.826.0.1.3680043.8.498.79008681898807131624328548092986371327","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39275838957651450909114440721012584183","series":"1.2.826.0.1.3680043.8.498.27430678348280622722276579569008663188","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39375796600811714870169173423682043709","series":"1.2.826.0.1.3680043.8.498.15594001162536018135542477247821207272","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402","series":"1.2.826.0.1.3680043.8.498.94088523047596279334511096043792912419","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39507184510584030713933330702725991027","series":"1.2.826.0.1.3680043.8.498.65683119241441620232671593277956891852","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611","series":"1.2.826.0.1.3680043.8.498.59853821380168052099489499213896251297","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39642276506855996181597769215091644995","series":"1.2.826.0.1.3680043.8.498.13306376038042703442478071760738485661","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917","series":"1.2.826.0.1.3680043.8.498.95123370568078933068598148297101944590","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592","series":"1.2.826.0.1.3680043.8.498.11013862647945118995152311807233506975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39954567501513957418422043089587980092","series":"1.2.826.0.1.3680043.8.498.48689332110162255440799136861204797283","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39969071491383654382076256213867635377","series":"1.2.826.0.1.3680043.8.498.68472185615684985634577766675738978529","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40041164441539690764994039690598305675","series":"1.2.826.0.1.3680043.8.498.13383888212045272725159464089845443048","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40154708689850542672289658249059660788","series":"1.2.826.0.1.3680043.8.498.28613289983395672961099780372189064222","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40239104167848425290754606013326180693","series":"1.2.826.0.1.3680043.8.498.10321362786209717535669252778379574447","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40257126480600927758308319831518876543","series":"1.2.826.0.1.3680043.8.498.95905329339158493145394893532948223183","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40308838286515017547707130085914948230","series":"1.2.826.0.1.3680043.8.498.10254199456670070400172004123551929578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818","series":"1.2.826.0.1.3680043.8.498.10450593028127771881719976733382103216","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40436886205373732563782477802719203679","series":"1.2.826.0.1.3680043.8.498.32087370442101308528498384349998340993","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40487147942341753806704765355921322161","series":"1.2.826.0.1.3680043.8.498.25432360075991334732337087707270021272","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40547250250566982489293057895661861722","series":"1.2.826.0.1.3680043.8.498.41856431174256977739201708769924922352","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037","series":"1.2.826.0.1.3680043.8.498.32294815986241189233910309454012506307","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40627043074325458726412346974457616938","series":"1.2.826.0.1.3680043.8.498.12138397179588503855766490119442272781","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40703974473944295875248011843170371394","series":"1.2.826.0.1.3680043.8.498.10074907724122468767765060075160612194","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40723835064354784133289444204866484476","series":"1.2.826.0.1.3680043.8.498.11280238824949571761802970991063413595","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40828786094337186697222449293183309391","series":"1.2.826.0.1.3680043.8.498.39363165183676433582495067617041128058","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40898220266193841898869905714214454387","series":"1.2.826.0.1.3680043.8.498.10615191289786772180127001051704421276","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40967435550546093348221485942169879116","series":"1.2.826.0.1.3680043.8.498.15546040320868786626038653438374347482","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40994616246264625946610553907227975893","series":"1.2.826.0.1.3680043.8.498.39626163178438023645080773465990705447","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41108421820714108307078614688022098388","series":"1.2.826.0.1.3680043.8.498.12326882012817398653011973581308471477","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212","series":"1.2.826.0.1.3680043.8.498.77190770549776586301241723160321465538","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41254472410580125134596211010943842213","series":"1.2.826.0.1.3680043.8.498.37407304328647389759764967335325862957","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41326575785634778198143222323166756210","series":"1.2.826.0.1.3680043.8.498.33150118058569713040300577326827793578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41382780862853765983216810855531499640","series":"1.2.826.0.1.3680043.8.498.11638620410303303001811570976398880632","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41440206520769979754790425253631809212","series":"1.2.826.0.1.3680043.8.498.12171728636223339153746880140971033521","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41587640208769641326416798167574152049","series":"1.2.826.0.1.3680043.8.498.98924069695181160094416830518112224446","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41605033773891953796694219242501252567","series":"1.2.826.0.1.3680043.8.498.45216752245787168433348135963095824027","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41623512571866040054960002151226127775","series":"1.2.826.0.1.3680043.8.498.67661320429290212989445761026899428235","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41662013720606493874198348776219875496","series":"1.2.826.0.1.3680043.8.498.12007106519684682440822683250096390140","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41696746845418651508748386392327996143","series":"1.2.826.0.1.3680043.8.498.12359306880269931579846655510208871515","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41717437729988689273718049162153247404","series":"1.2.826.0.1.3680043.8.498.13363071576670530227091719801291640575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41769275806689876192405091865340286535","series":"1.2.826.0.1.3680043.8.498.11010179874744075474240305053217270252","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683","series":"1.2.826.0.1.3680043.8.498.53896870119642370674342836256768434859","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41856137734822068230231773222820851271","series":"1.2.826.0.1.3680043.8.498.11744847833875650463259596689433989103","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41990779875873214154237496750949660480","series":"1.2.826.0.1.3680043.8.498.23637413006274253200504108537779807818","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42023788139360998656558751921918860808","series":"1.2.826.0.1.3680043.8.498.68067007265633660982839074818031065136","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42057349781448469704191384429694247725","series":"1.2.826.0.1.3680043.8.498.22239547516182772793672789354220097596","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42162585695624963017992042571207363427","series":"1.2.826.0.1.3680043.8.498.12261766702228825948914026285809869719","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42225165085995728538373532053227662192","series":"1.2.826.0.1.3680043.8.498.11388574511462185102378735263008272260","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633","series":"1.2.826.0.1.3680043.8.498.57765211809367396202162590867678866725","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42414269167296543219685897036829241885","series":"1.2.826.0.1.3680043.8.498.52946074053232527354101596503993592196","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876","series":"1.2.826.0.1.3680043.8.498.25395001047441323014558756794088120093","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42580107838226807862930203257414265276","series":"1.2.826.0.1.3680043.8.498.71960465090376842290219404972189035509","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42609322631348207494943622999125822017","series":"1.2.826.0.1.3680043.8.498.10116304832745128212628048203262986996","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42719592405241698450757802131459694665","series":"1.2.826.0.1.3680043.8.498.84735097096117550849768153143604555496","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42778524243264061817587398263842942394","series":"1.2.826.0.1.3680043.8.498.62603322073199615302226281522585863974","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214","series":"1.2.826.0.1.3680043.8.498.10779781833261359604879583885727533877","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339","series":"1.2.826.0.1.3680043.8.498.10242333489462409639931468061076863219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42942844751304929338846069579289911684","series":"1.2.826.0.1.3680043.8.498.11388557033882805833070303516525544252","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42990542863216044717944418712887800295","series":"1.2.826.0.1.3680043.8.498.82965617178080147198708102755915255000","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43046713539641995841029654207663845887","series":"1.2.826.0.1.3680043.8.498.62485273154790598285033438142945229107","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43127139928509033152155587499980767364","series":"1.2.826.0.1.3680043.8.498.13051584613832351246443921674237765289","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.43256282639827216310036033378453939750","series":"1.2.826.0.1.3680043.8.498.80368776335407672330887465322647367102","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43325226037168674077736336919956196684","series":"1.2.826.0.1.3680043.8.498.13084558074149905797431385478851595907","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43351215447181735764667371462573550267","series":"1.2.826.0.1.3680043.8.498.10459360829898188957672817058277043084","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43432499267536761263202158662502957684","series":"1.2.826.0.1.3680043.8.498.41672174561866286546929293154242945656","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43442933855807685981061557274166869461","series":"1.2.826.0.1.3680043.8.498.46544196769124529155674466903851797962","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43587191909319947256183198986671106743","series":"1.2.826.0.1.3680043.8.498.10967543785577894717040936811768710760","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43709699920591091261137846976272625163","series":"1.2.826.0.1.3680043.8.498.11253050832409393889626523751003658022","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43797037142729611453626414920554246615","series":"1.2.826.0.1.3680043.8.498.79564077268699258003978250387401716862","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43967752580489865323542041346927244370","series":"1.2.826.0.1.3680043.8.498.11211466512337820733032613684207233436","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44149403030093858908988179411336839587","series":"1.2.826.0.1.3680043.8.498.16513390873885589614724418437263866319","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44261256084247541720465769177956875383","series":"1.2.826.0.1.3680043.8.498.22196225336398392954638541855781616125","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44323131330689365651417881352766725366","series":"1.2.826.0.1.3680043.8.498.67340940015635996196998489512487508398","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44365814642869065241166313231424697474","series":"1.2.826.0.1.3680043.8.498.11131607393059253208448684148117349466","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727","series":"1.2.826.0.1.3680043.8.498.71713063348524022816903869057769600379","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569","series":"1.2.826.0.1.3680043.8.498.29656932758429639884826027502637714745","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44629238055006732101555855930071163428","series":"1.2.826.0.1.3680043.8.498.51287701793167814309669476089775146858","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530","series":"1.2.826.0.1.3680043.8.498.10125903227803478701129539319895874908","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803","series":"1.2.826.0.1.3680043.8.498.12387648861854820362985517945531478264","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791","series":"1.2.826.0.1.3680043.8.498.34410133400863425113932135007474049145","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44879493336213912007673875382158420811","series":"1.2.826.0.1.3680043.8.498.77757822396573023080717641045946739258","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010","series":"1.2.826.0.1.3680043.8.498.44616733692555217636792984144739838589","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45059917085750212913588683242522799083","series":"1.2.826.0.1.3680043.8.498.30644791285804552207042363602336200937","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45082393649917309927267478072569185694","series":"1.2.826.0.1.3680043.8.498.10833455296829138851597361388239020361","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45207812477654339877341538582090218293","series":"1.2.826.0.1.3680043.8.498.28957024987859670386693017068575444550","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45273393220765922863913956738614783632","series":"1.2.826.0.1.3680043.8.498.70117049278191551569360563837392228147","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45316453179223822297459860764883030492","series":"1.2.826.0.1.3680043.8.498.57929343334271560743267797210135406582","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45332185607602495786497298658937575959","series":"1.2.826.0.1.3680043.8.498.91259594630538749916213052339027918448","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45368532788186860828867663606651818835","series":"1.2.826.0.1.3680043.8.498.17696489087888680499237352685551955557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45406212529450909512649737062964080190","series":"1.2.826.0.1.3680043.8.498.52229212315896122321448078148406665545","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45463321406810606039579747001339703552","series":"1.2.826.0.1.3680043.8.498.46860826667982728788243308401669504462","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45504348071514081636909771148066956410","series":"1.2.826.0.1.3680043.8.498.10932995796154723393614655318975049915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45630051398709774279703407401383545642","series":"1.2.826.0.1.3680043.8.498.12002802440863757776953964940904575888","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45755198489211010290512228906807170657","series":"1.2.826.0.1.3680043.8.498.10593488711066451443698760079528470313","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45811964949115542204306831813385709206","series":"1.2.826.0.1.3680043.8.498.86323458031975595093802952242249050271","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45868806544202339567395643235910857549","series":"1.2.826.0.1.3680043.8.498.46821447623314009579340787258466380143","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45968957856865486437256472516713628614","series":"1.2.826.0.1.3680043.8.498.32487617485839375993248724519436539878","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397","series":"1.2.826.0.1.3680043.8.498.10291632104391532612227134181336655022","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46052469424267861803032899866428547156","series":"1.2.826.0.1.3680043.8.498.25290536932163484205591104991004626891","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46222691720198967161306520058359099856","series":"1.2.826.0.1.3680043.8.498.47499058030747191185224501701799997114","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46259494895672269475940484605821837768","series":"1.2.826.0.1.3680043.8.498.84271951417752430155883698568044421381","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46297226877524853395345229615300658523","series":"1.2.826.0.1.3680043.8.498.11654012003989326990449327599640523057","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46309540159823359540686782651264853894","series":"1.2.826.0.1.3680043.8.498.72104183779196527264713602286002355633","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46427879286435198611282055405541286766","series":"1.2.826.0.1.3680043.8.498.10752128828088140680408584096213354308","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46506161284114284997220751358509505380","series":"1.2.826.0.1.3680043.8.498.43299955150578416874720182505723693714","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479","series":"1.2.826.0.1.3680043.8.498.29532686309768444449872558266350951804","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241","series":"1.2.826.0.1.3680043.8.498.29042970259734963082888981176274150177","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46776367027887983105817631582518625639","series":"1.2.826.0.1.3680043.8.498.51170803576419176243354568428883605520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46842734152337184482152392783260161845","series":"1.2.826.0.1.3680043.8.498.12498877996849761744224515997669561791","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46893956584698923808118623450380169212","series":"1.2.826.0.1.3680043.8.498.21621349189556712551159070418990170134","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47004674612174010017653097242531940431","series":"1.2.826.0.1.3680043.8.498.26109988953413552297262786877156977660","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029","series":"1.2.826.0.1.3680043.8.498.47963401083152584918523226090823115662","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47080537724499166054609508776778189902","series":"1.2.826.0.1.3680043.8.498.18259829811596124361148635120228492385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47239511009810375185594384849101241004","series":"1.2.826.0.1.3680043.8.498.32863534199780667634018158313103557350","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47267174205472073784818634988669693642","series":"1.2.826.0.1.3680043.8.498.13282666580748334629961163840841140497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47339093185444044578247608022398494955","series":"1.2.826.0.1.3680043.8.498.49333474409006287578960938625470965266","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47465512474718256570933149642267926825","series":"1.2.826.0.1.3680043.8.498.42833914834945349229797525255530591339","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47531034194300620953276025401018051246","series":"1.2.826.0.1.3680043.8.498.14125873870023468442562261350438955754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400","series":"1.2.826.0.1.3680043.8.498.92464053013299249343643308323458300698","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710","series":"1.2.826.0.1.3680043.8.498.13063880277364995250041081090402109837","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47784065888386216103165400551121771174","series":"1.2.826.0.1.3680043.8.498.47138658445090985884214102064084236350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47870074144891988520855535250618228390","series":"1.2.826.0.1.3680043.8.498.10006846626751284159332285422987139118","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47949251060660360953300340380324578835","series":"1.2.826.0.1.3680043.8.498.79310377975048016176518963025632200476","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48068031587855513429324313927339963541","series":"1.2.826.0.1.3680043.8.498.91682052939073668535927648551851441033","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48308485987218296338173512060246238691","series":"1.2.826.0.1.3680043.8.498.15924496392525688503249382969845240469","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239","series":"1.2.826.0.1.3680043.8.498.29355335106065699355930513842423485014","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48417903971145834309474934717837250776","series":"1.2.826.0.1.3680043.8.498.66494020485767343881767722178613012956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236","series":"1.2.826.0.1.3680043.8.498.28752766479505763489028537924415760461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48564780460694033783288540237012068889","series":"1.2.826.0.1.3680043.8.498.70849905288648444552721197331717702724","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48697800450463192944610588533290320545","series":"1.2.826.0.1.3680043.8.498.24411134638040619329648990221935947032","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48780598610511011488578963416186905560","series":"1.2.826.0.1.3680043.8.498.11151071934815409155328374536679903067","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48818931767017230678539144135307491171","series":"1.2.826.0.1.3680043.8.498.91421098834852478720939798732398375308","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48872083192774100826667097552356058756","series":"1.2.826.0.1.3680043.8.498.11150076952638037404212933792851452910","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48933020429865163393792899740045803587","series":"1.2.826.0.1.3680043.8.498.11132822496616964286603191951939196488","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49025249926859976858200443939208207124","series":"1.2.826.0.1.3680043.8.498.12311344902890598588654770704454801018","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49048739684287632681242054218807611262","series":"1.2.826.0.1.3680043.8.498.16157778163535029227076159474622228519","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49094276335794811195117175603556929507","series":"1.2.826.0.1.3680043.8.498.32272180637884212608343547266133429981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49219586198038249030222134813685536295","series":"1.2.826.0.1.3680043.8.498.28109192989612980334933993736247999388","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49347024975371539869871751159907613638","series":"1.2.826.0.1.3680043.8.498.11022932223748132478475714854014070992","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49394404966674633152360206835416205330","series":"1.2.826.0.1.3680043.8.498.11763166767502702278007714151661899238","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49481811111224238761880576486796603139","series":"1.2.826.0.1.3680043.8.498.11956509073408998836685136529298595509","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49495644703047055149965226028097259207","series":"1.2.826.0.1.3680043.8.498.21209185315806127690769649466126872000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513","series":"1.2.826.0.1.3680043.8.498.38021093066816134221073666513556292228","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49660659440268002826070581802625582554","series":"1.2.826.0.1.3680043.8.498.78540269801527120175305875619580735404","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49818874332345742217581267858748819921","series":"1.2.826.0.1.3680043.8.498.55003952459689024049404351736477132515","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49850672111501136862025523789202033809","series":"1.2.826.0.1.3680043.8.498.21226117104406785484689676556842578550","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49866252311077397175807461727885923657","series":"1.2.826.0.1.3680043.8.498.18923266181608290837326127445409816002","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49887758195164927372912584780941789021","series":"1.2.826.0.1.3680043.8.498.10641039076705724840850567566778181933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49942556484410764813681696157580687854","series":"1.2.826.0.1.3680043.8.498.10446154915012575465331915267010323068","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50069224337753933811759198830086956116","series":"1.2.826.0.1.3680043.8.498.43369870036306270545644658015786022966","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50089015780087008332657793589955395952","series":"1.2.826.0.1.3680043.8.498.12929755044024048575370244206476642090","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50144715912379960437506930683683218333","series":"1.2.826.0.1.3680043.8.498.12854810061402211868166583319830091576","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50215409013618353346619153290049205718","series":"1.2.826.0.1.3680043.8.498.29758483246761679209486692262283791311","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50248717336781045512984550029203786086","series":"1.2.826.0.1.3680043.8.498.71531823989276758165331987858551456913","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50287512028986928372350279452197816661","series":"1.2.826.0.1.3680043.8.498.11167960371902581117403777530867229229","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50392110746914984873009735588879049321","series":"1.2.826.0.1.3680043.8.498.89078407514714031205012190805863930504","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50458745240631577290185672298591447264","series":"1.2.826.0.1.3680043.8.498.13217222805239364368672945153034319288","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50501020658943435430736717390350999395","series":"1.2.826.0.1.3680043.8.498.11362648711608348406048475555209180304","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","series":"1.2.826.0.1.3680043.8.498.12205463388882864789078909770673610236","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50667444467788034101157821576803778774","series":"1.2.826.0.1.3680043.8.498.11991498974727608833372834459106382694","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50687743970686088872471288288960579306","series":"1.2.826.0.1.3680043.8.498.20131777518321215096631464787908187696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50707407856055788475528375847872674241","series":"1.2.826.0.1.3680043.8.498.20233408566619568739155865816950994952","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50780438941220579448947609376330661130","series":"1.2.826.0.1.3680043.8.498.63899944800860034582719889561066669839","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50813123035221058777203670043892828582","series":"1.2.826.0.1.3680043.8.498.10394907151890306169495376508751915233","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50844227724977857739867646104945090414","series":"1.2.826.0.1.3680043.8.498.42820193944892248316818637812270889600","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50880891504121357724276091096406248845","series":"1.2.826.0.1.3680043.8.498.71338176779212764144955678112836742685","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50969691234820675847736765735133138324","series":"1.2.826.0.1.3680043.8.498.61877742053169611540605895321757204699","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51006813205605862503064447831381009425","series":"1.2.826.0.1.3680043.8.498.42038167025348624022730622708921580218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102","series":"1.2.826.0.1.3680043.8.498.68020908272736472159943670855014160147","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367","series":"1.2.826.0.1.3680043.8.498.64689304481117082040551911951538828085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51351386796317851725461803790273870995","series":"1.2.826.0.1.3680043.8.498.93726039761649478295274473648786509267","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51426696939581135481763379246368970360","series":"1.2.826.0.1.3680043.8.498.11893732387470280974464876066930374509","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51469564445249084310991187819410005207","series":"1.2.826.0.1.3680043.8.498.48046382359222601170373103544558092755","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51581188134109925758379838961317006116","series":"1.2.826.0.1.3680043.8.498.17759272266179354975388739101946887448","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51684117251092766010524179062007524257","series":"1.2.826.0.1.3680043.8.498.20120815116931082766720813121595457462","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51770131897828080663537020812721042704","series":"1.2.826.0.1.3680043.8.498.11757507768755227842830538753431873683","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51809092313496476873962641559638828608","series":"1.2.826.0.1.3680043.8.498.97792130861152356581305415611223856460","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51828180600648126726057100018010765613","series":"1.2.826.0.1.3680043.8.498.12076013011025274251971985536268719158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51867133375333949773527521703402111601","series":"1.2.826.0.1.3680043.8.498.65751042235229116816640129759366012876","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51917043703891091519740084517167861965","series":"1.2.826.0.1.3680043.8.498.70700038308036757685458740918791584013","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51981019698056120061499970572334688555","series":"1.2.826.0.1.3680043.8.498.86821232904196797613392639388119422294","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52010826440192227660457107121818389761","series":"1.2.826.0.1.3680043.8.498.11315255786368649423000283895823106000","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52059522418267239652374290810994711032","series":"1.2.826.0.1.3680043.8.498.81786487739787069798937386884134232379","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294","series":"1.2.826.0.1.3680043.8.498.93908436056411444186693339943723977228","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52138967568492774495485454357106269286","series":"1.2.826.0.1.3680043.8.498.10147248953663331628552894266420806404","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52204390785696944340286127542242341202","series":"1.2.826.0.1.3680043.8.498.11721940801082625877550198010398791315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52390908285903728334239836699232920065","series":"1.2.826.0.1.3680043.8.498.61220774517164658673338538741187891192","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52435631723314823200374985065756182593","series":"1.2.826.0.1.3680043.8.498.16189858906256822633822306034391637273","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52475562989926669452280424331791927661","series":"1.2.826.0.1.3680043.8.498.39065757396444777942430430320221683093","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.52520288400853832269613213204540385687","series":"1.2.826.0.1.3680043.8.498.76259907200602977691807910133715472219","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52636981391578580436180193211113340410","series":"1.2.826.0.1.3680043.8.498.86967676539123145215533337242918752391","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52745530063404662086111315503240395535","series":"1.2.826.0.1.3680043.8.498.61169900317316804184818293556310611732","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52846023545870016653608198538437574931","series":"1.2.826.0.1.3680043.8.498.89292347431865255429705985247231143875","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52937412433120578046842194545406797236","series":"1.2.826.0.1.3680043.8.498.42041060196909726635026828705126149781","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53025489296610571044246428509154950855","series":"1.2.826.0.1.3680043.8.498.95214721646826708158720693660974583961","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53080172184945962593713560671746700445","series":"1.2.826.0.1.3680043.8.498.80850470655415246748312428896346566628","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.53114802640679819441521055029024037716","series":"1.2.826.0.1.3680043.8.498.99010765620654796828970410213076929803","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53156118232564605912649705207438612782","series":"1.2.826.0.1.3680043.8.498.51313354734160276277285269666488484063","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53174810138171412044410050984734412146","series":"1.2.826.0.1.3680043.8.498.70491356790861828328319403299215186742","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53271710112621372496324855809391580130","series":"1.2.826.0.1.3680043.8.498.51019077351832620554300010316866835302","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53337684515627962545396495541158882262","series":"1.2.826.0.1.3680043.8.498.41554468574760235089349929159727188991","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53355961816756477601843499985089590263","series":"1.2.826.0.1.3680043.8.498.12979136123941104523472023603848418037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53393741562201846141435914434030220216","series":"1.2.826.0.1.3680043.8.498.10789155880195718818121806715435815351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476","series":"1.2.826.0.1.3680043.8.498.12161311395632993293125729526134485579","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53587920256280027683523152568832253423","series":"1.2.826.0.1.3680043.8.498.87035473995621406361302336538158655551","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53737349894831365159124471910871800583","series":"1.2.826.0.1.3680043.8.498.46155027986658676975615158756633491768","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53792159922324410857755054939229546555","series":"1.2.826.0.1.3680043.8.498.11309993371646943970624784857279720880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53893586200869613140666806653041177558","series":"1.2.826.0.1.3680043.8.498.11662175972157576496097094522766008171","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53963759852017695930808272438606800422","series":"1.2.826.0.1.3680043.8.498.97188670163617692947868485591502416737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092","series":"1.2.826.0.1.3680043.8.498.11514573965312681026370137143912454812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54100786607660749952814251186461344633","series":"1.2.826.0.1.3680043.8.498.11734336000951207506154411160400085205","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54132814503202891434380002378663763655","series":"1.2.826.0.1.3680043.8.498.78296454656977393877367317082779192298","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54164004416963296439322252627774426375","series":"1.2.826.0.1.3680043.8.498.12321258643475544728122921935116105938","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54270039578867689669466339980682684126","series":"1.2.826.0.1.3680043.8.498.49724356452476726386330211236910087941","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54316764984549863227549326352573342232","series":"1.2.826.0.1.3680043.8.498.11318160183080341539558636478754368230","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985","series":"1.2.826.0.1.3680043.8.498.79721747810511567405106660492095420673","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54448516923758211534546401490623824012","series":"1.2.826.0.1.3680043.8.498.84711926984239747202167814960254730984","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54482848766921329667469114743547203929","series":"1.2.826.0.1.3680043.8.498.17234926580009197333639351252282636725","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54539703415639765348239790227512479583","series":"1.2.826.0.1.3680043.8.498.86413821224936741428149115251333046119","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54628782412849994943720963251113534589","series":"1.2.826.0.1.3680043.8.498.75583201843492364072800165165095153360","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54689086194769265957947122616025330642","series":"1.2.826.0.1.3680043.8.498.69828282909876468301575636485219917392","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54699196872730350880681018674893216245","series":"1.2.826.0.1.3680043.8.498.44735312525062521498272305391321619586","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","series":"1.2.826.0.1.3680043.8.498.10460750967890626240281590186349431211","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54833384793167149128371137537527194757","series":"1.2.826.0.1.3680043.8.498.33254794194802912277586997722240216263","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55008913229055942530957507434366806883","series":"1.2.826.0.1.3680043.8.498.53840302265354817745078417656999620036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55022631713888448062201243978634363406","series":"1.2.826.0.1.3680043.8.498.88630720389848193582528157861324367746","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55067956175569778132589128971795053431","series":"1.2.826.0.1.3680043.8.498.78083328256136004172878020588186879012","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55183153701332792661581741475876132309","series":"1.2.826.0.1.3680043.8.498.13278841317503958516247817040053887575","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55332521421041708438748125666498042763","series":"1.2.826.0.1.3680043.8.498.87690851966981610535545817275966914802","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587","series":"1.2.826.0.1.3680043.8.498.10651889650079784026044753343456345122","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55415479446253262335685494509470808516","series":"1.2.826.0.1.3680043.8.498.14211785745941230142766650956182999824","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55583505147281684100202399148099429721","series":"1.2.826.0.1.3680043.8.498.51368028051808190736318829032961705773","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55687452977471862155042703116192295932","series":"1.2.826.0.1.3680043.8.498.37957409760550641558022054725471973215","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792","series":"1.2.826.0.1.3680043.8.498.11619616976146383592178609956570979768","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55771181380353653474896884649540028856","series":"1.2.826.0.1.3680043.8.498.59211316596902864246340233969120928397","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728","series":"1.2.826.0.1.3680043.8.498.51653489828105901957527277063098951201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55887747667305814945719986974538814207","series":"1.2.826.0.1.3680043.8.498.18624956303530587175397379136103215759","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55930031395651221279170435293900769500","series":"1.2.826.0.1.3680043.8.498.56537216409438351368489711785653711807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659","series":"1.2.826.0.1.3680043.8.498.68301641174868634080488853815207939362","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56045687004558569005240775213510192805","series":"1.2.826.0.1.3680043.8.498.77926510113017967767861991552347728985","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391","series":"1.2.826.0.1.3680043.8.498.32961253014558149634475770125613235212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56156923369480999484275191793049573490","series":"1.2.826.0.1.3680043.8.498.18008564429307804197842920043866165956","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56188634971310070481285525830793231849","series":"1.2.826.0.1.3680043.8.498.68468840382757606133329197909193477353","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56269695574661037525483516531194514456","series":"1.2.826.0.1.3680043.8.498.12712166077256945284538824203155932320","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56428314239264997930645427543149285770","series":"1.2.826.0.1.3680043.8.498.72985985357997686466651827988843792513","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56497268548664917337130798701015887461","series":"1.2.826.0.1.3680043.8.498.72580628806359217317695776871533064141","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56513276339820682030546599321915304265","series":"1.2.826.0.1.3680043.8.498.35748041848415859300931494010189369583","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56575942633137496743464648679775042979","series":"1.2.826.0.1.3680043.8.498.59309568439075714274470844181365117322","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56715311294415711253166312546942356882","series":"1.2.826.0.1.3680043.8.498.10146062361924542671529063152424956688","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56762592839620371588520593573846414860","series":"1.2.826.0.1.3680043.8.498.98129622460894334389266520383126182831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56798070549054124854685511360777559033","series":"1.2.826.0.1.3680043.8.498.41950554828765663333180907347187345177","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56839374698050163738039760889156066445","series":"1.2.826.0.1.3680043.8.498.10527451562657279934260499322722190857","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068","series":"1.2.826.0.1.3680043.8.498.10589258535438399062796780172737115901","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56915079210789596803972894081756325026","series":"1.2.826.0.1.3680043.8.498.14276377374594265690201863375200543502","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56945039212126810522718038626526579591","series":"1.2.826.0.1.3680043.8.498.78861758373352579779134855111487885489","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56993108275828399240508695615212151102","series":"1.2.826.0.1.3680043.8.498.12199149601302001967595573299098693993","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300","series":"1.2.826.0.1.3680043.8.498.13377701463417985398191636717670014325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57091203959244495164081406481825915856","series":"1.2.826.0.1.3680043.8.498.61513659200126070126654940188998932228","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57239450717434070615919278253974598533","series":"1.2.826.0.1.3680043.8.498.73412249209787520439382638792107501641","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57348281565196196401055821640494904475","series":"1.2.826.0.1.3680043.8.498.97901663158399834421443181033865357537","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57491375928585600499323136123872536597","series":"1.2.826.0.1.3680043.8.498.24361189682595621485649381877074516653","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57703608122587911055931213603182679454","series":"1.2.826.0.1.3680043.8.498.13301376644621675778646114501597169184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57775389384191131322519457853517177210","series":"1.2.826.0.1.3680043.8.498.49145413527911932598141448540567088020","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57811199787891897795884073594875380873","series":"1.2.826.0.1.3680043.8.498.11350982630196653994935352027805013045","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699","series":"1.2.826.0.1.3680043.8.498.12243302333248604116573935512731637540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57882532838957577543516026386452055977","series":"1.2.826.0.1.3680043.8.498.87065290887270006068249092331010001312","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57951268100539350409054562078476694936","series":"1.2.826.0.1.3680043.8.498.84831215317195643161270811841397118633","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58143227319316521737029358930629086887","series":"1.2.826.0.1.3680043.8.498.37363672605548524635220050862579935374","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58250779958866595505585700517927225230","series":"1.2.826.0.1.3680043.8.498.12082642116294597362590344354080295418","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548","series":"1.2.826.0.1.3680043.8.498.62993034304453904604924573677330911785","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58346459085553680879151588730895136332","series":"1.2.826.0.1.3680043.8.498.29720216239855236799575233895970744047","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58404766256608483217059706069436050741","series":"1.2.826.0.1.3680043.8.498.52098228800474391345283121348070032449","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687","series":"1.2.826.0.1.3680043.8.498.10102657421748772181920483019773111910","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58522891982092784622515590012748600644","series":"1.2.826.0.1.3680043.8.498.12768312452936639032505954723566915449","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58585441284202143792468863402617212167","series":"1.2.826.0.1.3680043.8.498.32130048553479504468465827978959266212","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58634494571833713296574414948455471101","series":"1.2.826.0.1.3680043.8.498.93953264214683515854274455995021884133","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656","series":"1.2.826.0.1.3680043.8.498.46828273462983774432213928747616250692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58739535411760425418941874065260802744","series":"1.2.826.0.1.3680043.8.498.11524692855668501193437190097905944438","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58768083117076186082335984587845964130","series":"1.2.826.0.1.3680043.8.498.19512391434365046339823938486916852574","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58811528407321581206563412079643094538","series":"1.2.826.0.1.3680043.8.498.71616153375607002769150078840515524936","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58857477075095368361468658035369975572","series":"1.2.826.0.1.3680043.8.498.20646038858592555186222993500341281244","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530","series":"1.2.826.0.1.3680043.8.498.54450284315070506389024331139874184611","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177","series":"1.2.826.0.1.3680043.8.498.82236127060871793224495432548823965831","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305","series":"1.2.826.0.1.3680043.8.498.17568228210008974949465689824961363281","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59126324546257448818397467078224944768","series":"1.2.826.0.1.3680043.8.498.13017867667165553396007947401145482044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59262240088132401543519466539885561578","series":"1.2.826.0.1.3680043.8.498.56134333245371099663962982619302065584","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59420153636832938628088507776889305823","series":"1.2.826.0.1.3680043.8.498.30288549815461656305838197032901206552","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59449702487184611096113620550399616423","series":"1.2.826.0.1.3680043.8.498.48032426515418196953907042356021491364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710","series":"1.2.826.0.1.3680043.8.498.38075577006651268414942662802300197372","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273","series":"1.2.826.0.1.3680043.8.498.74003595952032072412001119164504242501","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59680220055968160813801840287391339651","series":"1.2.826.0.1.3680043.8.498.12831377263102999999361056439687265008","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228","series":"1.2.826.0.1.3680043.8.498.48105186927843118060657358004760737756","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59835677513721120155185666132462195647","series":"1.2.826.0.1.3680043.8.498.12347553106626464881968568722506310858","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59923507410103964137819195926587517759","series":"1.2.826.0.1.3680043.8.498.67562553441066946185285764639779819729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59939977812549570931951749553467050226","series":"1.2.826.0.1.3680043.8.498.91930476890967668832806205706792482115","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59954207362160100069708115719895503197","series":"1.2.826.0.1.3680043.8.498.62600607422323790851475180038903444709","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101","series":"1.2.826.0.1.3680043.8.498.22146564323189332367311530977120194380","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60163612617486071760358965933890738162","series":"1.2.826.0.1.3680043.8.498.89548291157335136822329101869104211602","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60227430483444032531479844365716485643","series":"1.2.826.0.1.3680043.8.498.67250801866435724195229837589293460145","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60339775486026531260673620094004244044","series":"1.2.826.0.1.3680043.8.498.66461536089320846430291096531149817940","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60394858470917657607846893765818799720","series":"1.2.826.0.1.3680043.8.498.18858834741538714886853289900785219432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796","series":"1.2.826.0.1.3680043.8.498.27184683570152585303677760515729974516","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60465293103574559546982440125572490190","series":"1.2.826.0.1.3680043.8.498.12491736630196227468586365216318978509","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60477027832057450004004942199870813886","series":"1.2.826.0.1.3680043.8.498.54282762281956425318188232401394621049","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60582762894748063612925320003677861384","series":"1.2.826.0.1.3680043.8.498.73242261294459647349454068451348500496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331","series":"1.2.826.0.1.3680043.8.498.11198437071229260795013226124573912093","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60723378470101669902158229125336455520","series":"1.2.826.0.1.3680043.8.498.84872436487908266174683696640171031302","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60840220723824894619895264673401129352","series":"1.2.826.0.1.3680043.8.498.10913773047073790504809242518701269609","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60965286164497194740320124904525623873","series":"1.2.826.0.1.3680043.8.498.75412901425349593658482022394776911010","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61043690805522768840878574610198420558","series":"1.2.826.0.1.3680043.8.498.75493381247022181788777852006894063922","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61140619849833544411463750382539885141","series":"1.2.826.0.1.3680043.8.498.59895923095489424874677560277240107215","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61220609061901632500124675031032586841","series":"1.2.826.0.1.3680043.8.498.20067580693263035417862065477732035918","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61238657014751597736608285205361041675","series":"1.2.826.0.1.3680043.8.498.10605886642552974781117529358469507683","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61333047594780344399535845209036589563","series":"1.2.826.0.1.3680043.8.498.39955818992257795369652746197579419085","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61435096069921474761812731150911059491","series":"1.2.826.0.1.3680043.8.498.19159852753525900249365441169863961022","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61518377417979517706237284901970943470","series":"1.2.826.0.1.3680043.8.498.49775188754784417277292296218006201750","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.61554384977916135091630433680780524386","series":"1.2.826.0.1.3680043.8.498.10803965868662859696394825319541107371","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61581513083114643407028382481113097015","series":"1.2.826.0.1.3680043.8.498.75323083352757110071695519870499587515","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61797111379749329977228618245440305623","series":"1.2.826.0.1.3680043.8.498.28089945269410556393145923128011693073","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61864939963243675140060416958055914981","series":"1.2.826.0.1.3680043.8.498.58443329222919708249348834255270486013","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569","series":"1.2.826.0.1.3680043.8.498.19032935397356062721572728429462433090","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62030472630193758412620945408364583008","series":"1.2.826.0.1.3680043.8.498.11846647082029251559012315255234070547","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62154969360158054310282857093665911261","series":"1.2.826.0.1.3680043.8.498.71369721486275003821811031135466848400","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62205799963299902160268565319924399775","series":"1.2.826.0.1.3680043.8.498.45178786555424190880929564372254418064","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62251470468640645490157592046417355534","series":"1.2.826.0.1.3680043.8.498.12036690169154659079064932380973744533","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62329283530383340787096585492217632179","series":"1.2.826.0.1.3680043.8.498.12113554077383978666542828724589943090","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62400625562782184774973349983498962743","series":"1.2.826.0.1.3680043.8.498.84089251348902468964176207382932405934","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423","series":"1.2.826.0.1.3680043.8.498.42088692264623788581840990186297141548","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236","series":"1.2.826.0.1.3680043.8.498.18194661389225439596589076323105764635","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983","series":"1.2.826.0.1.3680043.8.498.13028465092536707248894452561135886108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62776347582771469289738234607517546504","series":"1.2.826.0.1.3680043.8.498.15550642845076069008515741187141347994","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62918046703989259691005447883746340916","series":"1.2.826.0.1.3680043.8.498.18234042265995401488689603628572511288","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62928513795250694194589804784951257414","series":"1.2.826.0.1.3680043.8.498.10765316083295720734095967085237842637","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63002559681883387414876870926128159682","series":"1.2.826.0.1.3680043.8.498.92998924954214560046069053863234372162","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63068687840516335034631353955761036465","series":"1.2.826.0.1.3680043.8.498.70384734067274388996581629815121305961","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410","series":"1.2.826.0.1.3680043.8.498.10645552537271899912431373465748459775","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316","series":"1.2.826.0.1.3680043.8.498.13277003188560614538003298895444944271","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63204735160524958513470115761010950181","series":"1.2.826.0.1.3680043.8.498.47624477857636226650919019604015128799","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63223220827342079068491460847713457238","series":"1.2.826.0.1.3680043.8.498.94266146351985729313200261158969451124","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63242029756172180508167933519465902827","series":"1.2.826.0.1.3680043.8.498.97673778748061976777762235901257537123","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63320397279666280420359587173167195895","series":"1.2.826.0.1.3680043.8.498.93474800335529789313147356306439348558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519","series":"1.2.826.0.1.3680043.8.498.78069691998995103433855879057128617147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63439188800533762543950349022148547855","series":"1.2.826.0.1.3680043.8.498.60426367783160955377702413471764796986","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63519003265636561965405570006747767668","series":"1.2.826.0.1.3680043.8.498.10544247591549418878395226360027938898","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63539809431911516912857504094611479480","series":"1.2.826.0.1.3680043.8.498.79817062498614870478597933036319922403","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63556690547679257097888724096967682731","series":"1.2.826.0.1.3680043.8.498.47288003673946983207703043621108977526","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258","series":"1.2.826.0.1.3680043.8.498.91769100149163118586875381282823957006","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63683477631564232018930819747238111809","series":"1.2.826.0.1.3680043.8.498.50235883352055161963216295295075820218","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63797588614544518196205411126336672334","series":"1.2.826.0.1.3680043.8.498.23993228475285857664708431307294110227","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63894147741855851067411708837856142566","series":"1.2.826.0.1.3680043.8.498.93788012685412213403308358302883507385","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64022112049842890696609934970222024490","series":"1.2.826.0.1.3680043.8.498.33182328540963800244465748903730501699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64034149345689391514924451724232196567","series":"1.2.826.0.1.3680043.8.498.71514182472260592483182091136783828025","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64109231850522212435708176045583779359","series":"1.2.826.0.1.3680043.8.498.43320620226749204901419630826990430488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373","series":"1.2.826.0.1.3680043.8.498.14916029409049153878790150011338044935","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64283618749539713756220539395944025878","series":"1.2.826.0.1.3680043.8.498.83940835335605839007009645522186343158","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64381606512330581256470279413868645789","series":"1.2.826.0.1.3680043.8.498.95000231071137877695427189800962335800","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64408775259838960958354835366765941676","series":"1.2.826.0.1.3680043.8.498.95827815628970638085416302997999112944","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64443149121726771900576109439951701105","series":"1.2.826.0.1.3680043.8.498.56516361927217933027899907427539946491","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010","series":"1.2.826.0.1.3680043.8.498.11132651848598435765910491586608293225","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64606794953061143951972249525094490665","series":"1.2.826.0.1.3680043.8.498.59831321824210785697691020674281972083","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64663285742626599570990862360902141188","series":"1.2.826.0.1.3680043.8.498.11810473262230298212206477377973963603","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039","series":"1.2.826.0.1.3680043.8.498.81574602288546285059782020305861974509","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64897285598346101578442697515294908341","series":"1.2.826.0.1.3680043.8.498.10083662364948490189619823703424674634","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64950600385134699009390197520158979183","series":"1.2.826.0.1.3680043.8.498.67632925968219948304543952978497568054","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643","series":"1.2.826.0.1.3680043.8.498.46579829652376930947749859378805653323","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65124952546314712352837396207287384405","series":"1.2.826.0.1.3680043.8.498.31875363548313786336603774074203461810","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65187850119346448625449915281726338455","series":"1.2.826.0.1.3680043.8.498.77608747667977697588085966558224985689","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65244223225926362862970496735749098549","series":"1.2.826.0.1.3680043.8.498.11213296701775130216492758691773697998","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816","series":"1.2.826.0.1.3680043.8.498.16231392397773664120365961190529518915","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65403984120528711726317048984986599537","series":"1.2.826.0.1.3680043.8.498.27555151041345516324135247398005699827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65524487168129064460609645228098420719","series":"1.2.826.0.1.3680043.8.498.67537847655496693912638524669982003257","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65547889329285432771702699287910809921","series":"1.2.826.0.1.3680043.8.498.10175798581885684200467490664905620301","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65607807464576681752987746458209389847","series":"1.2.826.0.1.3680043.8.498.27416101867555840463268928746427308428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327","series":"1.2.826.0.1.3680043.8.498.56213382323594894691466276669580439777","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65756786850574169314934708606155033765","series":"1.2.826.0.1.3680043.8.498.18110391809995942915703978303756044448","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65788206282522750507559682036750692505","series":"1.2.826.0.1.3680043.8.498.69961180546432025748542029401873306204","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65809115193576546782858254430325338674","series":"1.2.826.0.1.3680043.8.498.56829627798653213410775896917882242068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65896962360688978581417748865609785541","series":"1.2.826.0.1.3680043.8.498.73670680916106155925679770740048917487","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65939700606136711089565313916789162980","series":"1.2.826.0.1.3680043.8.498.12593148731639857200651022975267414247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66020213859389864645118554257523478921","series":"1.2.826.0.1.3680043.8.498.30418924875214199255407671754384949204","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398","series":"1.2.826.0.1.3680043.8.498.93248061954026560833033153514589483255","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887","series":"1.2.826.0.1.3680043.8.498.30961318189257492916956425361530479280","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66199248385068685539872840826351152652","series":"1.2.826.0.1.3680043.8.498.59628367200743960349158159409928546544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66287049728466409037095442705228040084","series":"1.2.826.0.1.3680043.8.498.97963626897294502853699831535941774181","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66305771884507759587182077473863996177","series":"1.2.826.0.1.3680043.8.498.38315244217262258380294979870112927043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66327751482076404294387513492434577338","series":"1.2.826.0.1.3680043.8.498.10151705117440282768723321288599557220","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66401771232064170830621215861092532421","series":"1.2.826.0.1.3680043.8.498.26354946991400667600556515008985634117","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66489634638847834550694493585147280754","series":"1.2.826.0.1.3680043.8.498.12405802931795205765868757513032133236","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66640124635222298792902508374800337726","series":"1.2.826.0.1.3680043.8.498.33015739889805041358257179456105658506","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.66694519343321893230666089565893349853","series":"1.2.826.0.1.3680043.8.498.45434640657520982642976588107285811845","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66841103065389483583108279900721522084","series":"1.2.826.0.1.3680043.8.498.43579676002521571296232740452299124682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66960891947495601371173426277430301127","series":"1.2.826.0.1.3680043.8.498.92453083385644021307088053022501214401","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67001329390153622359718581388215312710","series":"1.2.826.0.1.3680043.8.498.37057821200411117614339244183569084932","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67157948144868269956661697924341234924","series":"1.2.826.0.1.3680043.8.498.75410862133443248060678443830475548498","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774","series":"1.2.826.0.1.3680043.8.498.45473142305062485127856603480622246355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67238774585857531338533947812568490258","series":"1.2.826.0.1.3680043.8.498.25979179982243413397631207472207982708","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67374476726034608591738910363595695123","series":"1.2.826.0.1.3680043.8.498.79393086884376184829261604910097100565","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67397756161011684282778294948252031539","series":"1.2.826.0.1.3680043.8.498.10652117894528513687286244520313386625","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67468397709097947104298147948850473142","series":"1.2.826.0.1.3680043.8.498.36789025804716126939252928439200498676","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67498362978754615554767658151787188674","series":"1.2.826.0.1.3680043.8.498.12556389596317049931980418068213540723","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234","series":"1.2.826.0.1.3680043.8.498.12537044509644757605595003497605900524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451","series":"1.2.826.0.1.3680043.8.498.56194611687455488226700378555889173678","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67690090498828405347380696553164435363","series":"1.2.826.0.1.3680043.8.498.84889752217395965346145917211333095953","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67739091465243228820766982459725849814","series":"1.2.826.0.1.3680043.8.498.63206310871336511928872982502332650393","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67772676243889350087068112324015156848","series":"1.2.826.0.1.3680043.8.498.82075329312902559140507667379797678675","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67817378785531423675449308236680249899","series":"1.2.826.0.1.3680043.8.498.11089904712859283931858399330207208829","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67875980316053973061252844698884229168","series":"1.2.826.0.1.3680043.8.498.90511870677017494015026156722126457977","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686","series":"1.2.826.0.1.3680043.8.498.93063230799480820390464183661578432761","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67958551704692248657133951054036903879","series":"1.2.826.0.1.3680043.8.498.11272033137183498155471435765152767060","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68042422993686716523227090971667037309","series":"1.2.826.0.1.3680043.8.498.31983756592277722302653087083136989101","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68098431505945078581285587720599762771","series":"1.2.826.0.1.3680043.8.498.11193536677832213247620655480781601466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68115220080063728900711167735074276147","series":"1.2.826.0.1.3680043.8.498.43700493823504164461123555883041343699","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606","series":"1.2.826.0.1.3680043.8.498.58434302343002978719224489580240432515","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68168197665046786209369894696801397179","series":"1.2.826.0.1.3680043.8.498.37455864723963103288038733569737831064","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68236154531625015457518732674625182872","series":"1.2.826.0.1.3680043.8.498.34933097698585501840122402629656938250","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68263925314265871578642679583159577645","series":"1.2.826.0.1.3680043.8.498.87879732019072998353602846990716665243","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68328371080087298795966933114044350359","series":"1.2.826.0.1.3680043.8.498.10678051814403059304189101956371413817","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68434116759606759586218589635360502038","series":"1.2.826.0.1.3680043.8.498.29941809457544187987539879782119004790","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68442883836038453562548037358230921748","series":"1.2.826.0.1.3680043.8.498.99794845830208186585796590984378123308","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68504220056564878621959394439671931747","series":"1.2.826.0.1.3680043.8.498.73540242461191250812646859309355703332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462","series":"1.2.826.0.1.3680043.8.498.72177283734970100713122927680390625627","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68593524008570334427931338225473487643","series":"1.2.826.0.1.3680043.8.498.13094809319194150034176903123094560290","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68668620295929607459355154952035421306","series":"1.2.826.0.1.3680043.8.498.24423254443828559544786519275199482403","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68744926735490285461495285547312308522","series":"1.2.826.0.1.3680043.8.498.13290341826561633894758199057392847619","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68799355386870083063665227055340582612","series":"1.2.826.0.1.3680043.8.498.12932116267354179578824369138903760946","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157","series":"1.2.826.0.1.3680043.8.498.10161381035240214551499258258065290947","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960","series":"1.2.826.0.1.3680043.8.498.13300483737661434464502084777711642093","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840","series":"1.2.826.0.1.3680043.8.498.31221726774335595548186162311204064285","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.69077351372476086930472412978307038819","series":"1.2.826.0.1.3680043.8.498.69019954416931212845688892294638774102","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703","series":"1.2.826.0.1.3680043.8.498.35154789773882438895752037627964905064","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69260280362382086729793018979234418276","series":"1.2.826.0.1.3680043.8.498.12885289243912759875412944055343148631","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69311065409015951283858403030798356475","series":"1.2.826.0.1.3680043.8.498.11989408033844847960376383704457279670","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69411539430142524612470544041263824359","series":"1.2.826.0.1.3680043.8.498.19066568083695680394346414681413574477","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69429384137362531712760113705595820981","series":"1.2.826.0.1.3680043.8.498.82366966695051419751810080698431337022","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69514709065182531331987458626496800737","series":"1.2.826.0.1.3680043.8.498.14132567251985401960918884154460197678","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69632034454628807663252084651278084839","series":"1.2.826.0.1.3680043.8.498.99716827925331238821233940616644814424","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69654898267139720912940475772303404772","series":"1.2.826.0.1.3680043.8.498.12205028413762635875817948533464405394","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69717507929047897681405659943474469018","series":"1.2.826.0.1.3680043.8.498.53178218555130574368909582462432033199","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69755849784902271124335765927125722483","series":"1.2.826.0.1.3680043.8.498.62053587725425706510497258201564193602","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69846596572083911408018130524077120494","series":"1.2.826.0.1.3680043.8.498.31112845405018415174714970009950587393","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69954533310505596128507234525019777805","series":"1.2.826.0.1.3680043.8.498.10444593442885241053099863268327561494","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70125113032990356507120856030706104508","series":"1.2.826.0.1.3680043.8.498.10326415437405658980451000269329389988","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70212141059019236877414201247791694101","series":"1.2.826.0.1.3680043.8.498.70364528993490044299430494178633065955","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70310162525433830894324495692973986063","series":"1.2.826.0.1.3680043.8.498.11738867776084815778225875949338278311","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70358024114589255428192032298530338260","series":"1.2.826.0.1.3680043.8.498.37940543021741917944472922106974684817","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70427947434642957729711329062828233047","series":"1.2.826.0.1.3680043.8.498.79143413803594135274752026879964441486","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70471859687546886677145171619638581757","series":"1.2.826.0.1.3680043.8.498.55756521332796503160671244751077017073","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314","series":"1.2.826.0.1.3680043.8.498.32466031233091181446159248295902619008","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70609649448269244608860767754947143395","series":"1.2.826.0.1.3680043.8.498.97645275162293575397932890279699789873","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70691305452244378255646875206864340458","series":"1.2.826.0.1.3680043.8.498.43077138775719195355134064277352537333","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70829434408808897601630953382239533220","series":"1.2.826.0.1.3680043.8.498.27370954885415550020028787037145338748","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70873330457307886117949106432465978541","series":"1.2.826.0.1.3680043.8.498.10917914432944815634831532850092668018","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522","series":"1.2.826.0.1.3680043.8.498.86569833234152366143443820388448409891","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70918221697435256352088727172904502068","series":"1.2.826.0.1.3680043.8.498.25041887098153856347905859357161387935","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315","series":"1.2.826.0.1.3680043.8.498.46084908599417112056373029892294245892","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70955936086115471632446529979621948261","series":"1.2.826.0.1.3680043.8.498.41284183441965523748098439401831984550","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71001339006150287453095055802545712130","series":"1.2.826.0.1.3680043.8.498.11646421773246336021238000181955568791","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71063550263564056105844586167361750567","series":"1.2.826.0.1.3680043.8.498.70148514493589891032048586019990535064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71180975285730441100082825359032677931","series":"1.2.826.0.1.3680043.8.498.64843580064388389357901554603129313269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71355457409240695394093233905149541215","series":"1.2.826.0.1.3680043.8.498.62230538901567687350070877175092375661","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71405492944866967669069984757332849184","series":"1.2.826.0.1.3680043.8.498.21212884728324040686026100556156210188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71622107620007706778976946784787365505","series":"1.2.826.0.1.3680043.8.498.13292708762447172782529198377545566086","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71660070729774772658965427776586988585","series":"1.2.826.0.1.3680043.8.498.52816907244996120452188461997381295554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71688651674041466985305836912343513997","series":"1.2.826.0.1.3680043.8.498.35449707386100049389946541384814271493","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71707317731065645278250956442336515504","series":"1.2.826.0.1.3680043.8.498.11754959025541302848131560165050272426","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71744523263023933066263506201344989825","series":"1.2.826.0.1.3680043.8.498.13046171353053745384930244639212007490","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71753603955749189961638324806586359888","series":"1.2.826.0.1.3680043.8.498.10384075452197857357867761484406056527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71814919885777053270239861047114794237","series":"1.2.826.0.1.3680043.8.498.87003320539702170106596904541917806743","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71836684300308056297405632410209248276","series":"1.2.826.0.1.3680043.8.498.13390231351373978322142658196882071542","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71899370657875960028369229847153071577","series":"1.2.826.0.1.3680043.8.498.92029456967290746805884212849898897120","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71985417286226511458068110832784023627","series":"1.2.826.0.1.3680043.8.498.11662442678563405760122392692477318963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72018602547183896629903902899903468287","series":"1.2.826.0.1.3680043.8.498.82298660180940091968809690990704537638","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72095214441332094906401331319197122529","series":"1.2.826.0.1.3680043.8.498.13316536609584584669674987777567255460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72182847112910012489603688436912469641","series":"1.2.826.0.1.3680043.8.498.65244931710597747196879017666785065323","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72191353854496806844040272237700346774","series":"1.2.826.0.1.3680043.8.498.10998825089451117824678710755631375359","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72252601583562621620422058467969183753","series":"1.2.826.0.1.3680043.8.498.65266215631659597961698059413326392226","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72270915206075693056344475438055463113","series":"1.2.826.0.1.3680043.8.498.13370985172850418448230470663199212439","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72342425990274562150756501476989691572","series":"1.2.826.0.1.3680043.8.498.22381091527749503825098291514415530840","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834","series":"1.2.826.0.1.3680043.8.498.80384241949196238416728201739034918910","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72453426317225885428514595399496580896","series":"1.2.826.0.1.3680043.8.498.94742480575082698523805531710221925000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72495677804641720406765749568114476096","series":"1.2.826.0.1.3680043.8.498.55538903764787013012410334725936834829","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72576069358772864331810785663326683534","series":"1.2.826.0.1.3680043.8.498.53599434291381455888422329524324464934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72653666963736486779382314125782779004","series":"1.2.826.0.1.3680043.8.498.12992442475096075529653431889712849072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585","series":"1.2.826.0.1.3680043.8.498.82851172455284618768349877822670756434","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72781608662300877077691853670862414309","series":"1.2.826.0.1.3680043.8.498.11437384028024338193058889656036024708","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72814437733065652700292896915085936230","series":"1.2.826.0.1.3680043.8.498.12548212258128632520987427227890819488","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72880034930990967339546757487847803284","series":"1.2.826.0.1.3680043.8.498.52551231505760181422359060021237888416","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72931900540220826595003388066347326492","series":"1.2.826.0.1.3680043.8.498.11470304787149216826065490385397335618","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72943191222961132416678221885831944435","series":"1.2.826.0.1.3680043.8.498.49003920555049705190745584349138083268","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73033521237848260228708170280586043408","series":"1.2.826.0.1.3680043.8.498.35657934791265757895585981270394589619","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73070182506490199787361298167985971577","series":"1.2.826.0.1.3680043.8.498.11809354913653326221794865797132843494","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73083786204651635096412213233334934941","series":"1.2.826.0.1.3680043.8.498.39397416221880642811163735307409677183","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73194579460722181989151410198309642266","series":"1.2.826.0.1.3680043.8.498.42954787679788576701850405246412133278","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.73205916973275504240156555900460720571","series":"1.2.826.0.1.3680043.8.498.87929125724733479288292489657953342541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73241519025825862373621891294281253158","series":"1.2.826.0.1.3680043.8.498.21873284249118906695110689517010968046","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73257122462858353041066181004123253930","series":"1.2.826.0.1.3680043.8.498.30204301724123748920313377484685239193","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73314900301724532218347713512381622074","series":"1.2.826.0.1.3680043.8.498.60626044696085585755403022958872427134","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73394992092536626285935384324191560182","series":"1.2.826.0.1.3680043.8.498.66680164344286469950397716935069032081","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73477249559479870799427355988138990349","series":"1.2.826.0.1.3680043.8.498.70174700810834803930473922536710802408","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615","series":"1.2.826.0.1.3680043.8.498.24185627522498470160664893715676328329","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951","series":"1.2.826.0.1.3680043.8.498.36615537033339492408423676664912459318","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73602922823389605007702521335136347605","series":"1.2.826.0.1.3680043.8.498.11917628152448493587884825628630776826","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73682608760611536443754406204042210115","series":"1.2.826.0.1.3680043.8.498.33436014535043212811554590899765264936","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73792361507751648386401969571717557389","series":"1.2.826.0.1.3680043.8.498.62045029214226027809156465863531742099","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73869775636181013203432845292462040922","series":"1.2.826.0.1.3680043.8.498.98815188404735933813248825505612212999","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73983081622010008601849687650985260980","series":"1.2.826.0.1.3680043.8.498.10914038255939092774341180556379414108","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74052473719853343027548507306186291134","series":"1.2.826.0.1.3680043.8.498.93101014537500183521522248803693783431","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74127742661271965093205833471645893520","series":"1.2.826.0.1.3680043.8.498.70050427946073135537164269806368927701","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74160604730133050784449689952695940558","series":"1.2.826.0.1.3680043.8.498.90594650246380221192619947199422285438","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74214829456846525506232367308573550186","series":"1.2.826.0.1.3680043.8.498.72424642450047131542594192077313144372","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74222284584657051737421776919599206132","series":"1.2.826.0.1.3680043.8.498.30336284545027348260106715926414494762","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74243564762473331581882860440560337725","series":"1.2.826.0.1.3680043.8.498.91789778597168289082063337026362543448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74274855596893595390880617896069524465","series":"1.2.826.0.1.3680043.8.498.12076044731343465919353653900854683907","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860","series":"1.2.826.0.1.3680043.8.498.63723275346850158441274261395643419312","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74394784271003789564066563841806954326","series":"1.2.826.0.1.3680043.8.498.99072092540318807442839474329430118815","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74471396469132211069109289423495141062","series":"1.2.826.0.1.3680043.8.498.98916021586303358322367556298725115907","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74622031396885197580453304563088863451","series":"1.2.826.0.1.3680043.8.498.55758965806491791137748397491439871794","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74714018385278253151175853998945654373","series":"1.2.826.0.1.3680043.8.498.44320650968782861044403957869459496155","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204","series":"1.2.826.0.1.3680043.8.498.82000133706494980144637539357651413316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74872419726042767543822107291861307866","series":"1.2.826.0.1.3680043.8.498.45078443524636868490207992609658588439","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74939269621045823332837094516056211606","series":"1.2.826.0.1.3680043.8.498.89575886757405228775030604828375281628","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74952769952682175467667812958859010329","series":"1.2.826.0.1.3680043.8.498.57476531181249737132050505431957281821","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716","series":"1.2.826.0.1.3680043.8.498.33175037533438804952879631219403198406","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75268429892526831752484062674699274309","series":"1.2.826.0.1.3680043.8.498.27733888368619725295443212573892103716","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75282577107802752393765861202996650638","series":"1.2.826.0.1.3680043.8.498.83630809257438667292130449138652448794","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75384600248630012126841890313149674691","series":"1.2.826.0.1.3680043.8.498.30446340117313927948925332074135327319","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75420186662442688955616630735303321846","series":"1.2.826.0.1.3680043.8.498.33753794008902252062377193224041065838","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303","series":"1.2.826.0.1.3680043.8.498.83548477920297455372096703070558819891","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75547515925579172661099437338637082554","series":"1.2.826.0.1.3680043.8.498.53379844123719420871125385286668732403","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759","series":"1.2.826.0.1.3680043.8.498.11566733817314434492587763761542971318","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347","series":"1.2.826.0.1.3680043.8.498.50108507105969417467201187993403170645","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75688484058613633923943882730405145535","series":"1.2.826.0.1.3680043.8.498.45683657919697351224148835029861865767","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75742195884147990442829552992649204078","series":"1.2.826.0.1.3680043.8.498.86629663978615250845226551144024835973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75760698509105388883811155432691667447","series":"1.2.826.0.1.3680043.8.498.35629950349429980089877975532443041705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75800418125597325281807386716279941068","series":"1.2.826.0.1.3680043.8.498.58072050334557303365317000149596152536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75926011938952404257302292495731660728","series":"1.2.826.0.1.3680043.8.498.48873199751227706417614547964619974253","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76012896432318288623173593280895772876","series":"1.2.826.0.1.3680043.8.498.13261628281554546656868103601582968798","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76065434429437758192282562351089010088","series":"1.2.826.0.1.3680043.8.498.39124831840138085556877842282580348730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76122127303562588545128053661609574837","series":"1.2.826.0.1.3680043.8.498.20089779260780487923500296703080005851","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.76167153566894354739332364376483514559","series":"1.2.826.0.1.3680043.8.498.11114247136853746156054961065779716303","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76242245340157474968574926079920787199","series":"1.2.826.0.1.3680043.8.498.38648942231302708338149882783871514210","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76350662767360412185091180512006419379","series":"1.2.826.0.1.3680043.8.498.48951641721181168463157892981045627059","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76514504104121634569792846980451896030","series":"1.2.826.0.1.3680043.8.498.47515843810588294070043096003473452606","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76543241221133987658552440969465372120","series":"1.2.826.0.1.3680043.8.498.10273168415186102908332976788231504149","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636","series":"1.2.826.0.1.3680043.8.498.31405281328700191565363081406934192637","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76680092665218514711300086707222109323","series":"1.2.826.0.1.3680043.8.498.10817483223356577160213445231211830700","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789","series":"1.2.826.0.1.3680043.8.498.30645725191324992481547413938622618918","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76862911430418639277808760660745763387","series":"1.2.826.0.1.3680043.8.498.12034997945035092597086684404071936315","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76914908131203373740345988718674850939","series":"1.2.826.0.1.3680043.8.498.10736315093068497910528660086880830836","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76993922254007307143291254174266755701","series":"1.2.826.0.1.3680043.8.498.53557542505997549449801890485341161966","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509","series":"1.2.826.0.1.3680043.8.498.28023367278171807139094491533496770222","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77061877382110597010150770981465581536","series":"1.2.826.0.1.3680043.8.498.11804319226999646874563309651435985038","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855","series":"1.2.826.0.1.3680043.8.498.99377292327043323184900631329171681588","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77215476384973989055525592556183934827","series":"1.2.826.0.1.3680043.8.498.10617153966080303206956655554595198888","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77261014467268552758826551926128217025","series":"1.2.826.0.1.3680043.8.498.31170434578225866618575626131670969602","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917","series":"1.2.826.0.1.3680043.8.498.93532039867332149733057145449301227864","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77299727102400166279263320763707637485","series":"1.2.826.0.1.3680043.8.498.86430061665535505501250740061953782970","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77319661699409997424800377883567027839","series":"1.2.826.0.1.3680043.8.498.21021816904284119428178963238360667196","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77370119801633353780677378448761665685","series":"1.2.826.0.1.3680043.8.498.71149197179783144358569569626084091582","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77416382267531567401698555072875486512","series":"1.2.826.0.1.3680043.8.498.30104701667579896444369864697667532584","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77433065090757862487482575794398458402","series":"1.2.826.0.1.3680043.8.498.90149977213100621732939023387481171331","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971","series":"1.2.826.0.1.3680043.8.498.28500225464005195330357793658890588068","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77524507552350448792970399203131998592","series":"1.2.826.0.1.3680043.8.498.12536148523062091476795401915897564117","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77546769004367585830455474984769277713","series":"1.2.826.0.1.3680043.8.498.42753658605739301027076067102474474128","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77593859518024366170735307975173961777","series":"1.2.826.0.1.3680043.8.498.73905051809455032170790310172576735119","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77626578713099735143745396709359607445","series":"1.2.826.0.1.3680043.8.498.79128104802182852834470932013347274466","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77735690782343511872544703464260892488","series":"1.2.826.0.1.3680043.8.498.10147795444151277772976219360072357467","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77753753872601674668756777373761660873","series":"1.2.826.0.1.3680043.8.498.83876882504334828318534957599403928311","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77800740811302686332282410929816056283","series":"1.2.826.0.1.3680043.8.498.10194179723779378797899379932366061061","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77823946901144772346251781172043480398","series":"1.2.826.0.1.3680043.8.498.65446571842218281758797691343556571026","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77893704208186903913532950495036889026","series":"1.2.826.0.1.3680043.8.498.55575706137099723690091131566298661702","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77933128806366660573360274958258917742","series":"1.2.826.0.1.3680043.8.498.93435326478599563896562827715787345099","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653","series":"1.2.826.0.1.3680043.8.498.44625635971565103849461471134689247361","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78083057157844664959843364584162587026","series":"1.2.826.0.1.3680043.8.498.21893789792855968893520644819255897402","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411","series":"1.2.826.0.1.3680043.8.498.44501769583743172214407804507327536993","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78236036694398253902871510456985685293","series":"1.2.826.0.1.3680043.8.498.45748648251197946600637287844267690283","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78282552802075313852355443512555224586","series":"1.2.826.0.1.3680043.8.498.78181317637477969377334702318909327167","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78420763928408680447797586163977368190","series":"1.2.826.0.1.3680043.8.498.82031987641448558748731167771711866143","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78475145549489837635803968565380247517","series":"1.2.826.0.1.3680043.8.498.56606402062863213762797399362334540752","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78521396404736679127326025240279747196","series":"1.2.826.0.1.3680043.8.498.12231840660708707763308896624440740363","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78646473961447523033955716877524794707","series":"1.2.826.0.1.3680043.8.498.35075507587015938414313962250575617849","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78674598286586525734293563394409166190","series":"1.2.826.0.1.3680043.8.498.10677479317607814532353073244078363282","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78757011872917563830995996191850541453","series":"1.2.826.0.1.3680043.8.498.11264087156793461594243140373111930400","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78808320605816811006382053060951793555","series":"1.2.826.0.1.3680043.8.498.11600616576501266090514740522132041831","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78853931928960361455025737444267199231","series":"1.2.826.0.1.3680043.8.498.23070521550719606485106450744833955037","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79021499290691125513421477204330532755","series":"1.2.826.0.1.3680043.8.498.52740698740121868101539374909046993156","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79077219037094025366585138112899760966","series":"1.2.826.0.1.3680043.8.498.11255993703470581697219105731116371159","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79117047831806895544174734229992359986","series":"1.2.826.0.1.3680043.8.498.10075012787048324306023820168721006946","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79165531588966776246377098630025444815","series":"1.2.826.0.1.3680043.8.498.21676668049071126965759942921489193083","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317","series":"1.2.826.0.1.3680043.8.498.79116287154220586195464520786363907335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79258497507223824227830886693749460348","series":"1.2.826.0.1.3680043.8.498.14427064458194729137435881588912903648","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79274948549030940175536603063980168192","series":"1.2.826.0.1.3680043.8.498.84502839634883102589019584889003824694","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79292862882980756453890778996691993351","series":"1.2.826.0.1.3680043.8.498.33294899683937455703608043752572828526","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79505118958475739396320647697583979079","series":"1.2.826.0.1.3680043.8.498.59131574285165914052755456526959101715","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79551456190098389081551755482042135791","series":"1.2.826.0.1.3680043.8.498.46297495675016978936408645711926516232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79582457162604674356422857305405255763","series":"1.2.826.0.1.3680043.8.498.79243957964868708759873405187865967165","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79636443202100988025435536780567446582","series":"1.2.826.0.1.3680043.8.498.92596996672773279242556192419627281044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79725520325008084984331823107016254775","series":"1.2.826.0.1.3680043.8.498.55437767063342025142296129763675287356","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79798779812052605703861699094268016306","series":"1.2.826.0.1.3680043.8.498.33559957994323452238912495340726004882","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79991838737793096019496013540529642900","series":"1.2.826.0.1.3680043.8.498.74747889114349610270638073997332348473","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80100785345864355815698147663964005920","series":"1.2.826.0.1.3680043.8.498.99323638971171814093049789501463100892","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80161127293548940216292842707683784128","series":"1.2.826.0.1.3680043.8.498.64096639544016150809824005424725822501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80207056225476761432712249807198836187","series":"1.2.826.0.1.3680043.8.498.12377984937793060485220183983609749210","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80275294885165896124489476449616436102","series":"1.2.826.0.1.3680043.8.498.85796252806425670445507148103074147138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80292580287849976694993628109721422478","series":"1.2.826.0.1.3680043.8.498.44493174746357665185363724680030049230","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80319546205646617603546887163539306146","series":"1.2.826.0.1.3680043.8.498.59388253108973531050886333076531218114","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80391378873630377472192017862302619947","series":"1.2.826.0.1.3680043.8.498.84184427111740960124940422935353479489","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268","series":"1.2.826.0.1.3680043.8.498.62813766153756972992989403420813685776","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286","series":"1.2.826.0.1.3680043.8.498.28205207786933309777978554820637175193","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80612964209397056653334784776523875647","series":"1.2.826.0.1.3680043.8.498.14068254637341147172867019200578324006","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307","series":"1.2.826.0.1.3680043.8.498.42392777333884830786644281676812540367","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80643480554795312280333659144941545925","series":"1.2.826.0.1.3680043.8.498.14432012861570019271138094593309342993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80671597537348112977780954910198682272","series":"1.2.826.0.1.3680043.8.498.11442645944879606999756025648574848093","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80735285867775404153100236162962472952","series":"1.2.826.0.1.3680043.8.498.97719206090121634386832089787727144629","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80758336123908522542787716219939751704","series":"1.2.826.0.1.3680043.8.498.92727678135443834505983274544767298650","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80876235930771882271600826613593202364","series":"1.2.826.0.1.3680043.8.498.32524605078687261234014764219261890982","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80886525412417948114478368374120704834","series":"1.2.826.0.1.3680043.8.498.36406069090593427841919922610548751087","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81018911907787774214591206085583545057","series":"1.2.826.0.1.3680043.8.498.41639959346748067367109996785694657843","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81031057654953109295788159826841315360","series":"1.2.826.0.1.3680043.8.498.88948597818012337244640158359167477197","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81109405367395078564887780335570099673","series":"1.2.826.0.1.3680043.8.498.10298690419190573645735602705319806031","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81248820934296461385324385812581970340","series":"1.2.826.0.1.3680043.8.498.73326878474940612527600507535696940526","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81327951282145199298572980264557740321","series":"1.2.826.0.1.3680043.8.498.40269546008002648800929538803647044112","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81347841116019279082226143308588516787","series":"1.2.826.0.1.3680043.8.498.12506317361825291063377427175139944064","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81462603621780919719466264356289554669","series":"1.2.826.0.1.3680043.8.498.73341490919573184799812164626315452655","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81466575994184158172641823163654418018","series":"1.2.826.0.1.3680043.8.498.67856873324260940777918960278854858840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81540134120507822899514611863725372563","series":"1.2.826.0.1.3680043.8.498.10007333964995826484716358858897372841","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233","series":"1.2.826.0.1.3680043.8.498.13321126994019818317755993107492731399","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81651132686162360431299931833372141617","series":"1.2.826.0.1.3680043.8.498.42373554791283824394352776941278669137","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472","series":"1.2.826.0.1.3680043.8.498.64998082693097953609029600267444478489","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81738571920712657790861120957086010772","series":"1.2.826.0.1.3680043.8.498.85748864540245146326521223816357695423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81793454171031578492222004629265937656","series":"1.2.826.0.1.3680043.8.498.93940000466321058184462420816180151649","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81797437102217585216278133168092577952","series":"1.2.826.0.1.3680043.8.498.60986070724986878747292491220412669859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236","series":"1.2.826.0.1.3680043.8.498.77496211030168448227100573543841167427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81927409938015158663782061022054347955","series":"1.2.826.0.1.3680043.8.498.65321952107851544099745236110796182211","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81974732979506931694079719601760393277","series":"1.2.826.0.1.3680043.8.498.40716748596899488748490486780036080624","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.81984200870681776955493738987078746894","series":"1.2.826.0.1.3680043.8.498.98337017554593669490287906558880597400","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82074883987134049971242727164506046476","series":"1.2.826.0.1.3680043.8.498.40330932826339924121039132056739203539","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426","series":"1.2.826.0.1.3680043.8.498.36194919842596510955209245388670337122","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254","series":"1.2.826.0.1.3680043.8.498.41453776462847737420755793667519107493","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82355456395534835058857499033494900716","series":"1.2.826.0.1.3680043.8.498.69106345857178534659060588499714713447","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82440268452598207215458064865523988222","series":"1.2.826.0.1.3680043.8.498.27618976484590124213072599837618912628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82498616123349854719801627024073383874","series":"1.2.826.0.1.3680043.8.498.63810019752701386902392014480429339885","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82548251499640077572026404183402934330","series":"1.2.826.0.1.3680043.8.498.29899465726532858804525475828486744345","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82582558624896662901163827145277490232","series":"1.2.826.0.1.3680043.8.498.75877669925357107017900878786996798478","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82655005222804459605894403147145992714","series":"1.2.826.0.1.3680043.8.498.82597143385827122977438785600307173462","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758","series":"1.2.826.0.1.3680043.8.498.98468939320829259978417816469672454721","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82902769135373284012762237744774967400","series":"1.2.826.0.1.3680043.8.498.47666158487996047725467912328415170289","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82967086493280898295669930903240677729","series":"1.2.826.0.1.3680043.8.498.59957770786475850426886924378002227010","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966","series":"1.2.826.0.1.3680043.8.498.10956898688790451040341513112018084600","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83055401642066749019091368358783700783","series":"1.2.826.0.1.3680043.8.498.94184901983654938070265493727732603017","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83108474538388106770920972885900596508","series":"1.2.826.0.1.3680043.8.498.13041923658224282265573316676798175869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83220896521302064187926748035734200732","series":"1.2.826.0.1.3680043.8.498.86650093216346141241806212373691377883","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83237706274929081175032033288272528919","series":"1.2.826.0.1.3680043.8.498.72009143853219762873024941491737381268","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83376388091077619281568190257019735638","series":"1.2.826.0.1.3680043.8.498.94552719977181739321910514313369583635","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83465081013175989426506371406463660936","series":"1.2.826.0.1.3680043.8.498.45879016245872161467743687497816427503","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83509478059288902959149676250911654986","series":"1.2.826.0.1.3680043.8.498.89116126151068379633097459199866156428","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83691358175810149975084604975065122220","series":"1.2.826.0.1.3680043.8.498.15223450556059808513268992552245338731","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.83747418172021065605877274971789603771","series":"1.2.826.0.1.3680043.8.498.55121003970028553862027611984974907430","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84007436672323587547274248257117331532","series":"1.2.826.0.1.3680043.8.498.98616955290063681311671981729639798982","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84059077072076289248966490696872414783","series":"1.2.826.0.1.3680043.8.498.37664835128584560297077779051965359919","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84062980592611347665739105270731842220","series":"1.2.826.0.1.3680043.8.498.29599807307280751683794072105043441972","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488","series":"1.2.826.0.1.3680043.8.498.96156752686964573586973693984772259671","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84238814030979264404768702945290704724","series":"1.2.826.0.1.3680043.8.498.76489574455843059179586804208900808193","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84419297518873367123595080276985526259","series":"1.2.826.0.1.3680043.8.498.13571094474282934367974627224519457400","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84483080389344581106129884869678629612","series":"1.2.826.0.1.3680043.8.498.54234347741256565356784159708919782223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84630493975910180537831353884518764364","series":"1.2.826.0.1.3680043.8.498.70457705312001387299396446372835178715","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.84661341697154104963062177064495044195","series":"1.2.826.0.1.3680043.8.498.10666740124861088611029915183399401569","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84748479740284893375192048321707057482","series":"1.2.826.0.1.3680043.8.498.12869137146851358776931931771663347916","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84855341377363129926967387137216814518","series":"1.2.826.0.1.3680043.8.498.28805127462509331733817972009907867606","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","series":"1.2.826.0.1.3680043.8.498.10234730915522311141420971962025744945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85031127696558971105465423134104223548","series":"1.2.826.0.1.3680043.8.498.28867171747975114017236595633706756078","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85095389649021071892478435298763928925","series":"1.2.826.0.1.3680043.8.498.50577589194754166963938938542568937863","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85194002080793317087210934629814349622","series":"1.2.826.0.1.3680043.8.498.92388163147247307112165897052932723067","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85294435004325211584698794441083866921","series":"1.2.826.0.1.3680043.8.498.62209313801260400687864127421056585487","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85341019006931436802294963713199308935","series":"1.2.826.0.1.3680043.8.498.35747407154620527218627871501507181618","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85391052076760686301971262244561025345","series":"1.2.826.0.1.3680043.8.498.20037074343623246005303284943102661606","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85416107585703628549041626609272707431","series":"1.2.826.0.1.3680043.8.498.70622940029345323879676343141861519504","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85464150757448712711832350617010924327","series":"1.2.826.0.1.3680043.8.498.12992984733496381551961559037629508310","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85499036441349866810703945289793057098","series":"1.2.826.0.1.3680043.8.498.68044827767799081572937527001298118252","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85602291032341007550376696695283794665","series":"1.2.826.0.1.3680043.8.498.59872436184221309660812447876459259180","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85646781189339988389190891868001374533","series":"1.2.826.0.1.3680043.8.498.40926508947516227467365047729545608553","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85687351346612079711706269264454944647","series":"1.2.826.0.1.3680043.8.498.19947391893023696730622359751892838453","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221","series":"1.2.826.0.1.3680043.8.498.49893436904175863101120702742306439555","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85735729179216019275685564112535593988","series":"1.2.826.0.1.3680043.8.498.48635603031405692168974485905052039947","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85849908712977285006159496192673293501","series":"1.2.826.0.1.3680043.8.498.64429864971055314770719700090747763451","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85879572979834912642178551098379145607","series":"1.2.826.0.1.3680043.8.498.36329657359205140854463659396278515440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290","series":"1.2.826.0.1.3680043.8.498.53416920592541912226331371042841973033","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85970222049966384156120176704985355996","series":"1.2.826.0.1.3680043.8.498.12758704938149954689594793071594895583","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85992037603186775437749979929724514304","series":"1.2.826.0.1.3680043.8.498.47236727135944736019984334543833447347","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091","series":"1.2.826.0.1.3680043.8.498.12581104722758394442748165850384663894","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86067849233330188988240710585112183308","series":"1.2.826.0.1.3680043.8.498.87541031377110409635053457453321840377","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86142930956005716169649168306759482915","series":"1.2.826.0.1.3680043.8.498.13143669967668328573010374900469059353","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86251314013737658351649446042801810374","series":"1.2.826.0.1.3680043.8.498.81913119155453543143927508721077896114","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86273853115722509980053190345047859057","series":"1.2.826.0.1.3680043.8.498.30601973622799115968186923190300363404","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86285715293767810771609523819042157072","series":"1.2.826.0.1.3680043.8.498.15194074121584753000927052341852297869","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86341139705884690862568425859303870601","series":"1.2.826.0.1.3680043.8.498.94815116565815829224926154471042565321","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86405969333062807387894608315897106350","series":"1.2.826.0.1.3680043.8.498.90385867400793527323765408683920730692","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86422343639522250286002195622435564347","series":"1.2.826.0.1.3680043.8.498.22404684360876336035826673676484465112","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86435890312227285216810145658548164773","series":"1.2.826.0.1.3680043.8.498.15557101642469816109240695337762281353","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86566584776908143877603164704838586097","series":"1.2.826.0.1.3680043.8.498.10141616667504148915268422810417973077","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86625876521226465945114246972542881955","series":"1.2.826.0.1.3680043.8.498.12789962617254909158403478522760123792","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86733565200815744326667230375642936291","series":"1.2.826.0.1.3680043.8.498.28737992266179111425351874951901725319","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.86881633316011036335786865282908198831","series":"1.2.826.0.1.3680043.8.498.69320418525937827171503970573469308386","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87009029107490862793898457518597022930","series":"1.2.826.0.1.3680043.8.498.54131453173718251602484545352401549675","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87046380372347615219261898797043742937","series":"1.2.826.0.1.3680043.8.498.95595248727651811721359713559367571961","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520","series":"1.2.826.0.1.3680043.8.498.12894903597444056914857129251003514789","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87198191797585172912979594638355685739","series":"1.2.826.0.1.3680043.8.498.91465768446102981871861367071095441138","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87222653734797522858506220026427900177","series":"1.2.826.0.1.3680043.8.498.11806670576643553087145001819020237905","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87241650441242408799533998101943024283","series":"1.2.826.0.1.3680043.8.498.86767507000037761029839825319480758497","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87267658933515453963224923798100651549","series":"1.2.826.0.1.3680043.8.498.13024759821532414941010416384995613157","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","series":"1.2.826.0.1.3680043.8.498.68272292264218910396961856501685863847","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87465202523430400780596885230492167155","series":"1.2.826.0.1.3680043.8.498.55305295157426121319714316590849277107","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87516159856486535118712846576943170757","series":"1.2.826.0.1.3680043.8.498.14983083009268847851717415059479781670","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87563449504927531224663601218159863518","series":"1.2.826.0.1.3680043.8.498.56282299025671023058643630874958938493","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87650297321648368205324227429768256110","series":"1.2.826.0.1.3680043.8.498.15966648716022679940788003643309543814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348","series":"1.2.826.0.1.3680043.8.498.92908716682482071335804193039057341334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87769640167791104899516835133527936885","series":"1.2.826.0.1.3680043.8.498.37834575012808663708255830116173653142","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87799106106199445436621174506197684035","series":"1.2.826.0.1.3680043.8.498.10890099088015365317680495451163647548","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87876800169766592603600598167358214111","series":"1.2.826.0.1.3680043.8.498.11818628183363456674948937482326731288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88022456207467042544403752144674777523","series":"1.2.826.0.1.3680043.8.498.12088270530793831161382588267080618557","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88070896604159829163161019117970440189","series":"1.2.826.0.1.3680043.8.498.72538141138037613245096515241403305038","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88114918358665488269761388157480523350","series":"1.2.826.0.1.3680043.8.498.97718061181650166984615821372374567323","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685","series":"1.2.826.0.1.3680043.8.498.30427265655095811817783086382011334830","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88439732365857773633852840383740478214","series":"1.2.826.0.1.3680043.8.498.89744006889126909137023023761109698484","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466","series":"1.2.826.0.1.3680043.8.498.11742799805820445193847689179737648692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88523737187772411911621088333701306485","series":"1.2.826.0.1.3680043.8.498.12738437102971023129353335979355648934","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88571985843649843469694556575325271560","series":"1.2.826.0.1.3680043.8.498.46771835536573851128848990622841770255","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88609906210780568637209291182572648571","series":"1.2.826.0.1.3680043.8.498.40836053550102648377673010032900284831","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88655485247489827200671636011965281184","series":"1.2.826.0.1.3680043.8.498.12151815026129672055842662297543673309","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88682169866098509464628948503453852928","series":"1.2.826.0.1.3680043.8.498.76127329275016586482481751078395632396","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88726804476694411363564819498296119763","series":"1.2.826.0.1.3680043.8.498.52299152708520353611671871075478014387","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997","series":"1.2.826.0.1.3680043.8.498.70910906875743778045929610545123014359","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88845193477394064535233691626377077328","series":"1.2.826.0.1.3680043.8.498.34981511111089449805959430901618726716","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88874647535130028956725641820977918615","series":"1.2.826.0.1.3680043.8.498.10555575703325287190969275761783419354","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88899519473612920868099376939780201067","series":"1.2.826.0.1.3680043.8.498.10884835004503941795991463947143409798","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800","series":"1.2.826.0.1.3680043.8.498.11061248166953059964701105014673035599","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89048907638162719840222508855779172366","series":"1.2.826.0.1.3680043.8.498.39533472144757892975282478246029488313","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89110271437785174288044757208973128908","series":"1.2.826.0.1.3680043.8.498.62018716226369535010620260506531423256","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89190511197564127079697151213577962649","series":"1.2.826.0.1.3680043.8.498.98018485122832897038407961215385262568","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89247482662611898098489676481780144186","series":"1.2.826.0.1.3680043.8.498.10312407932919120668921981217665499143","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89307235192406983202868051697765144920","series":"1.2.826.0.1.3680043.8.498.11687745101811297942848788408891991668","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946","series":"1.2.826.0.1.3680043.8.498.11580735692366186501390685933687226909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89357823844365374776758395347764601030","series":"1.2.826.0.1.3680043.8.498.87237757600814356363619379015007738913","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89445342796957655182687412965056270903","series":"1.2.826.0.1.3680043.8.498.13483502456971084658588395470116140170","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89569762088162031469299314053759758927","series":"1.2.826.0.1.3680043.8.498.49409892860037058572959872729527317610","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89584627944947813918297925813120247226","series":"1.2.826.0.1.3680043.8.498.42784882035384744939614619697591500902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89729486637365590877469535082675069395","series":"1.2.826.0.1.3680043.8.498.12504345164525907272058721660204761685","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89829843399218627892046362324057240535","series":"1.2.826.0.1.3680043.8.498.12193614395246345038022771337107697028","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89935384074170571164295050203650829722","series":"1.2.826.0.1.3680043.8.498.65085629822223134933550772032935178213","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89953731420879076053772747816096629231","series":"1.2.826.0.1.3680043.8.498.71823485538908357429956892406039459236","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90003472003278237920648198243464374280","series":"1.2.826.0.1.3680043.8.498.56103431114678465268734684730898961067","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90057130466053377217652879986591534745","series":"1.2.826.0.1.3680043.8.498.30105810453417909350614749183924561771","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90170179734833731758923964833173536172","series":"1.2.826.0.1.3680043.8.498.19812012461156187934436339625790375164","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90193654806390401129887447441469319421","series":"1.2.826.0.1.3680043.8.498.36684691011830782009263861672691117753","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90227414485305096825508911050013279704","series":"1.2.826.0.1.3680043.8.498.24588981968822714362346239896557245406","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90264300606047579088362539636050224478","series":"1.2.826.0.1.3680043.8.498.17928384674498826963779898154266856918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767","series":"1.2.826.0.1.3680043.8.498.12094995878175738580208251372899753847","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90313529150165069133283531436628039942","series":"1.2.826.0.1.3680043.8.498.18958899581592919281587875661179823059","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164","series":"1.2.826.0.1.3680043.8.498.12623791104654067959027521061861002483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90446842983860370183814559926724769347","series":"1.2.826.0.1.3680043.8.498.87554791425169923469696535646239895565","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90483774039833567637730989859822082272","series":"1.2.826.0.1.3680043.8.498.11570587382386087812865546262588679696","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90514150380296346197442897381548735293","series":"1.2.826.0.1.3680043.8.498.10604473055133273149151774796247402968","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.90573769367031841574114648773801425781","series":"1.2.826.0.1.3680043.8.498.12176139889509434951130748907630069123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680","series":"1.2.826.0.1.3680043.8.498.63482963825995204726134297820392991232","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90758414832541790868268922794274257839","series":"1.2.826.0.1.3680043.8.498.46153874061438526031938322859708952330","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90771070773751767226565553195436971014","series":"1.2.826.0.1.3680043.8.498.93893549927535961849648904435254420655","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506","series":"1.2.826.0.1.3680043.8.498.61658218918329993013102909516813325451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91064647744940506223536204418057554573","series":"1.2.826.0.1.3680043.8.498.73676698661079598805607207894460396340","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91122873808284731048016330734204992698","series":"1.2.826.0.1.3680043.8.498.68731028511941089241983696629901269138","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089","series":"1.2.826.0.1.3680043.8.498.90913540948969581132463718870717332410","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91206973544724011503639050238118945528","series":"1.2.826.0.1.3680043.8.498.85468604912020347655229321355315995472","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91230059913252604666971822137553690811","series":"1.2.826.0.1.3680043.8.498.13253381355522110590794922436955938438","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91323765268136520804399328350809179765","series":"1.2.826.0.1.3680043.8.498.70460732955894259968017992496652508467","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394","series":"1.2.826.0.1.3680043.8.498.97191915009073287261467823753909738143","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984","series":"1.2.826.0.1.3680043.8.498.86032361659667055392608606647765095501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785","series":"1.2.826.0.1.3680043.8.498.52749095849460196588130677813713342002","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91497176426480836856097456438014037999","series":"1.2.826.0.1.3680043.8.498.12921968613937351294073112536012728234","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91606395836371290680039618164703651751","series":"1.2.826.0.1.3680043.8.498.93015236021054681190470178884031371358","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91686669332744972709173289374472094988","series":"1.2.826.0.1.3680043.8.498.51680283785949667157500165000420649562","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91742054227008690570256869200116133922","series":"1.2.826.0.1.3680043.8.498.56827380780001508988526833983053752590","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593","series":"1.2.826.0.1.3680043.8.498.67653362891651822517845817697046144873","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91910585599088493175245713843727208369","series":"1.2.826.0.1.3680043.8.498.87469108885017732343816545181334174508","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91940000162714863613139612302783091395","series":"1.2.826.0.1.3680043.8.498.51858367042355966820568114860565427819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91991985280680856830060431668414014465","series":"1.2.826.0.1.3680043.8.498.75641273081924454933866890948865697233","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92071199603599654827155753204585082303","series":"1.2.826.0.1.3680043.8.498.13405446726346802090202898935773602145","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92222956609345084635188178384706987675","series":"1.2.826.0.1.3680043.8.498.12790991420664796821623258368244408848","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92285140961160393284537044926797259671","series":"1.2.826.0.1.3680043.8.498.49591269298695287630780643405035312069","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92386519544130078776672685639580497728","series":"1.2.826.0.1.3680043.8.498.11308487680616834626317656980156162091","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92410537691481397330646029810090178745","series":"1.2.826.0.1.3680043.8.498.74243763118970753834389206474439561602","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.92449204325697143336618138322280397765","series":"1.2.826.0.1.3680043.8.498.35758640376564846750508208181075505980","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92597065276819676534178545455668171085","series":"1.2.826.0.1.3680043.8.498.44949030617051703143706740150338281902","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92626490033376759309712417622692944095","series":"1.2.826.0.1.3680043.8.498.10306112669019925822296306999264646078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92655291659311409459093330375014445330","series":"1.2.826.0.1.3680043.8.498.12629288281741626188383805937315143682","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433","series":"1.2.826.0.1.3680043.8.498.60320422028331202523265248117792465813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92839397144213404004277739877191283401","series":"1.2.826.0.1.3680043.8.498.83119563715158751970459179214373044001","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92902692647544828045768164058889405369","series":"1.2.826.0.1.3680043.8.498.10642965759110860280654205173748389373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93054327536380380487760790221613186305","series":"1.2.826.0.1.3680043.8.498.40590250646909543190672186024486706836","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93104685852024334015651558514195715956","series":"1.2.826.0.1.3680043.8.498.10212051612126477490384560977205547306","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93177444420568709574758402836371200945","series":"1.2.826.0.1.3680043.8.498.13029560999668843281902300991355573552","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93188773180403601524488403580611534053","series":"1.2.826.0.1.3680043.8.498.80762101453077991639350335848305776483","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93301337627536436086612556500687157032","series":"1.2.826.0.1.3680043.8.498.81988624245965590770122638353071698525","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706","series":"1.2.826.0.1.3680043.8.498.69268597642480711940957758189393965718","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93385433312281352893457225415812880948","series":"1.2.826.0.1.3680043.8.498.31290427729728472257614568042102115039","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93623421864452751381778712192879121271","series":"1.2.826.0.1.3680043.8.498.53803019656714449935839402179839876488","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93718451128867963123946970296603639068","series":"1.2.826.0.1.3680043.8.498.28725769140622930098083929957675505091","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93722352425168280299901366261374293722","series":"1.2.826.0.1.3680043.8.498.51601288960315474687762408438152276425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93774667493652492119129601342627338680","series":"1.2.826.0.1.3680043.8.498.11788331906742866755360949121016518067","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93921993164883084178408030385007465064","series":"1.2.826.0.1.3680043.8.498.73235399263768777780495686179887410901","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93985093305024344873337249433396141838","series":"1.2.826.0.1.3680043.8.498.41204267314628424703592433520152178258","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94030816415316308210133175314873817171","series":"1.2.826.0.1.3680043.8.498.10247483317221578690430150192831941049","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94117540970936035429846288243624086693","series":"1.2.826.0.1.3680043.8.498.22118248635648893752981143165881977818","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94255124739558547668304022649099251744","series":"1.2.826.0.1.3680043.8.498.11923901515182510277294472906346327815","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94296943940827583101208235208463322140","series":"1.2.826.0.1.3680043.8.498.85682509836578029531646275116813748900","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94381282809153270156987123429739805206","series":"1.2.826.0.1.3680043.8.498.10567102468849839560979737494058810650","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312","series":"1.2.826.0.1.3680043.8.498.81238513949482991110216640426071715261","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94553664348118184200007654265863262684","series":"1.2.826.0.1.3680043.8.498.53753608720667242905303121168343302573","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94601583997736688762302974573903982304","series":"1.2.826.0.1.3680043.8.498.37638033628970872008663668858847550233","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94625454484290996254750481989036812516","series":"1.2.826.0.1.3680043.8.498.80773570850173541776889835133157159953","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94677297888135961827159679484207792618","series":"1.2.826.0.1.3680043.8.498.61967982759232501725130037568787203340","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94729634553835820944505159643192054175","series":"1.2.826.0.1.3680043.8.498.15424914578429205535083915070547192733","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94761755148413531477206243804832613511","series":"1.2.826.0.1.3680043.8.498.70406148617706490784288538750943927112","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472","series":"1.2.826.0.1.3680043.8.498.87310061282441705205910885328394403442","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94944218313788305917509271079443871442","series":"1.2.826.0.1.3680043.8.498.25239358560098660131109110275059837125","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94954489705497211843190734684710878341","series":"1.2.826.0.1.3680043.8.498.61185940944585658798988120110124492017","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95030266788619257057351879256576338162","series":"1.2.826.0.1.3680043.8.498.59993344953138931102668901204717213646","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95080449640993916851075185393444497308","series":"1.2.826.0.1.3680043.8.498.11353304001702558573298294648804928630","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95142886737239438773421088967802136318","series":"1.2.826.0.1.3680043.8.498.10000477131564453926848372798213249270","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95196756237527666678816095708949563380","series":"1.2.826.0.1.3680043.8.498.51588551017390676795340332424605064430","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95262231945171099723559613554350591139","series":"1.2.826.0.1.3680043.8.498.85153347264301775957424407568593974367","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95324318295506605557933083155792587297","series":"1.2.826.0.1.3680043.8.498.17912959879515996770450735371822811127","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963","series":"1.2.826.0.1.3680043.8.498.36236344117030528678128162382538381578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95464807346156547985763374990661923815","series":"1.2.826.0.1.3680043.8.498.57964960232336523011845134852370066253","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948","series":"1.2.826.0.1.3680043.8.498.31200373418774635433994734418906313503","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310","series":"1.2.826.0.1.3680043.8.498.78135625644505614805514506759814876347","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95586477246200285088746818821808017494","series":"1.2.826.0.1.3680043.8.498.47890053152792137761128810340618418919","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95621693348567815920675804584086926261","series":"1.2.826.0.1.3680043.8.498.17229947347876975651132025332411286279","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95653228095828585867177997067051877351","series":"1.2.826.0.1.3680043.8.498.47909343698113837871880603658739232349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95720534093121754146413060811354489247","series":"1.2.826.0.1.3680043.8.498.73646458242388788236405176157770373511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95728724960383464194568391302477328450","series":"1.2.826.0.1.3680043.8.498.50693358090017322689401496556310141013","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95850844945129153291759207212815927078","series":"1.2.826.0.1.3680043.8.498.41609637465948657915108683054382953655","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95882731821729917461275694847968503951","series":"1.2.826.0.1.3680043.8.498.94596794449423953090576739103260529071","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466","series":"1.2.826.0.1.3680043.8.498.67123608798819813175449846409698720711","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96054287426681123062488435620352206601","series":"1.2.826.0.1.3680043.8.498.85947819447886582244691863490635344275","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916","series":"1.2.826.0.1.3680043.8.498.12687145002197108396898450731381292045","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199","series":"1.2.826.0.1.3680043.8.498.90329165803497361417571658749000250156","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96316229168827165128604998083731481211","series":"1.2.826.0.1.3680043.8.498.12500557397847286688304642875678683529","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96425049974772570897047688166336162674","series":"1.2.826.0.1.3680043.8.498.11598672696827829517752117691491604543","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96510621665552630060821422211995212854","series":"1.2.826.0.1.3680043.8.498.19946550407577320614369143649162241292","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96561978771090020509310535265236331946","series":"1.2.826.0.1.3680043.8.498.66955923697534070352085755661006202677","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96564709711765982786409987515735443801","series":"1.2.826.0.1.3680043.8.498.46867767224708288615225261725085737471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96621557839848412555023112236627663339","series":"1.2.826.0.1.3680043.8.498.22815859205001731068686703895613493393","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96818533353374094273022322108861564926","series":"1.2.826.0.1.3680043.8.498.35410345699500894327897197263739813743","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96859317329616917022799512740280754327","series":"1.2.826.0.1.3680043.8.498.18842565802003680662093981855116887490","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96947237995559790704304892745226362883","series":"1.2.826.0.1.3680043.8.498.13331707818562992054120820404034811419","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97003157807630958179276831877591411623","series":"1.2.826.0.1.3680043.8.498.71329049553222208048538943277038623135","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97042446513400958172996637321308032556","series":"1.2.826.0.1.3680043.8.498.56323916502049690171035592694373974890","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97087958215554026148585108910609248523","series":"1.2.826.0.1.3680043.8.498.44065876311251021734600446022122774105","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97156518925820521351080132062400795929","series":"1.2.826.0.1.3680043.8.498.64841492746843093371611773736213835231","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97234348688223610934999923021953923903","series":"1.2.826.0.1.3680043.8.498.13030826860141949143201384510936206638","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97259977321685658337087695233264517545","series":"1.2.826.0.1.3680043.8.498.91173909879005616388938734082428621962","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97274847570547462056218828853253311223","series":"1.2.826.0.1.3680043.8.498.11551639437747726323825187583742075520","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97348112008590316355392924312704081684","series":"1.2.826.0.1.3680043.8.498.42372439479738242633784669273686432168","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97464332941382646555656038793589898504","series":"1.2.826.0.1.3680043.8.498.29059387952282622358079429160833660531","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97488501634729541923597957717971376373","series":"1.2.826.0.1.3680043.8.498.10242973867664012246006049941190811404","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97500632594529766269369977508352975364","series":"1.2.826.0.1.3680043.8.498.10209726568234009935051464275771080333","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97536230315600409983600607358927775480","series":"1.2.826.0.1.3680043.8.498.24848051499273874677843037512413445902","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028","series":"1.2.826.0.1.3680043.8.498.12179970650166968272663986814144233201","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091","series":"1.2.826.0.1.3680043.8.498.13388466006568704979238256617953118178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97705920250069816684861966612302390614","series":"1.2.826.0.1.3680043.8.498.41183518157246110038428106897732217587","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97759980486481979519068526243106370997","series":"1.2.826.0.1.3680043.8.498.47730761152818617620764059613661340802","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97849030682292314880463675622456900933","series":"1.2.826.0.1.3680043.8.498.35685899073300231496268108879582370055","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499","series":"1.2.826.0.1.3680043.8.498.81609906023834118013609796699208856249","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98027050915885651900992366068887550187","series":"1.2.826.0.1.3680043.8.498.37861541164004547717732461837552657646","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381","series":"1.2.826.0.1.3680043.8.498.74059415257269473036501171147182918805","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98081486779628580946589119341754353354","series":"1.2.826.0.1.3680043.8.498.11709982339742116481737894108414837047","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98112373403331748422888362310572528002","series":"1.2.826.0.1.3680043.8.498.61007601600470099121953938854148724501","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98146155044178593810385943365143374814","series":"1.2.826.0.1.3680043.8.498.32811399939595489548550698443205961760","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98239758242192048511912905088391835938","series":"1.2.826.0.1.3680043.8.498.15905559565818793882156015421784397346","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98315778542448798493600485116505234132","series":"1.2.826.0.1.3680043.8.498.81212289294690585293990567610053283976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98351891935507349137982199658103205287","series":"1.2.826.0.1.3680043.8.498.33597425682382622868077791548155741702","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98454206323401382140655387224146512225","series":"1.2.826.0.1.3680043.8.498.98440274254760510687660632317020920984","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98581537941208552065033680853629122764","series":"1.2.826.0.1.3680043.8.498.72964068698875321388975822271740051234","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98752901295449528283322564256444025625","series":"1.2.826.0.1.3680043.8.498.35882411650404635958892870273331142450","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98795677271204502833293873138219040186","series":"1.2.826.0.1.3680043.8.498.12022127372004239671461335856422355310","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98828753472895685940998888332947119084","series":"1.2.826.0.1.3680043.8.498.20620456124233467817883834013681702636","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98890851347757949328159691828768030366","series":"1.2.826.0.1.3680043.8.498.23938821005077743660489157826762525656","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98958651320983196982606333548522846934","series":"1.2.826.0.1.3680043.8.498.57293817763047086224372204679665327833","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659","series":"1.2.826.0.1.3680043.8.498.26297611965069209578322206839597161370","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99155175553749543768821558247962844052","series":"1.2.826.0.1.3680043.8.498.26239217436277568619016677488177468798","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99243902497374852166139435719549686908","series":"1.2.826.0.1.3680043.8.498.57243907673573292081252677002805586466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99303079292320085374389086609761974820","series":"1.2.826.0.1.3680043.8.498.71103253860611776587435100525803283290","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99414163787650487720542840540379270468","series":"1.2.826.0.1.3680043.8.498.67186687390706416103240943408253281919","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99444110025206148999881196086066868280","series":"1.2.826.0.1.3680043.8.498.11909317943703642764266639122290444087","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99454657594216441037514296544803013407","series":"1.2.826.0.1.3680043.8.498.82914395939851657368993919945583251102","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99526998617163591447503530185291775961","series":"1.2.826.0.1.3680043.8.498.95435245205115134599176639727000034815","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095","series":"1.2.826.0.1.3680043.8.498.66800547188021364535297656740745769947","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99708702196778120583644711299950439036","series":"1.2.826.0.1.3680043.8.498.63929390406093421141480628725922002640","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99725523560254732094432244556623376711","series":"1.2.826.0.1.3680043.8.498.50174723276820764851261348616601790593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99813269006669714758887909212386431934","series":"1.2.826.0.1.3680043.8.498.64167339835370193667326960744138590779","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99827460918477409069820779642161448917","series":"1.2.826.0.1.3680043.8.498.84111703816917031730631015620429748318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99880021280666757817587834481314366555","series":"1.2.826.0.1.3680043.8.498.12185957768992451405253311374381841608","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99939100657291954890286900729983287973","series":"1.2.826.0.1.3680043.8.498.73237121125256452563264987887442750681","plane":"Axial","fluid":1,"slot":"AX_FS"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1777
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
